# C0.0 WHICH DATASET — set this first, before running anything else
`DATASET` is read by Cell 1.4 via `os.environ`, so it must be set BEFORE that cell
executes. On Kaggle the simplest place is right here, in the first cell.

It is ordinary configuration, NOT a Kaggle Secret — secrets are for credentials only
(the wandb API key). Nothing here is sensitive.

Changing it re-namespaces everything automatically: run ids, the checkpoint directory,
and the wandb artifact names. RSNA keeps its historical identifiers untouched, so
switching to another dataset and back cannot disturb results already stored.
Cell 1.7 asserts all of that; if it does not print PASS, stop.

In [ ]:

import os
os.environ['DATASET'] = 'RSNA'      # <-- RSNA | VinCXR | LAG | BrainTumor | BraTS2021

# Epochs follow the dataset automatically (MedIAnomaly options.py): 250 for RSNA,
# VinCXR and LAG; 600 for BrainTumor. You do not set them by hand.
print(f"DATASET set to {os.environ['DATASET']}")

# C1.1 General used support functions

In [ ]:
import subprocess, sys
from importlib.metadata import version, PackageNotFoundError
from packaging.version import Version

#  This function checks if a package is installed and meets the minimum version requirement. 
#  If not, it installs or upgrades the package using pip.
def check_import(pkg, install_name=None, min_version=None):
    """
    pkg          : the name you 'import' in code (e.g. 'sklearn', 'skimage')
    install_name : pip package name, if it differs from the import name
                   (e.g. import sklearn -> pip install scikit-learn)
    min_version  : minimum acceptable version, e.g. '2.0.0'. None = any version ok.
    """
    name = install_name or pkg
    try:
        __import__(pkg)
        if min_version is not None:
            try:
                installed = version(name)
            except PackageNotFoundError:
                installed = None
            if installed is None or Version(installed) < Version(min_version):
                print(f"  ⚠ {pkg} version {installed} < required {min_version} — upgrading...")
                subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                                        f'{name}>={min_version}', '-q'])
            else:
                print(f"  ✓ {pkg} ({installed})")
        else:
            print(f"  ✓ {pkg}")
    except ImportError:
        target = f"{name}>={min_version}" if min_version else name
        print(f"  ✗ {pkg} — installing {target}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', target, '-q'])

# C1.2 Define required packages and check/install them

In [ ]:
# --- SINGLE SOURCE OF TRUTH ---
# To add a package: just add one line here. Nothing else in this cell changes.
REQUIRED_PACKAGES = {
    'torch':      {'install_name': None,           'min_version': '2.0.0'},
    'sklearn':    {'install_name': 'scikit-learn',  'min_version': '1.2.0'},
    'numpy':      {'install_name': None,            'min_version': '1.24.0'},
    'matplotlib': {'install_name': None,            'min_version': None},
    'pandas':     {'install_name': None,            'min_version': None},
    'seaborn':    {'install_name': None,            'min_version': None}
}

for pkg, spec in REQUIRED_PACKAGES.items():
    check_import(pkg, install_name=spec['install_name'], min_version=spec['min_version'])



# C1.3 Import the required packages for the project

In [ ]:
import os, time, json, random, warnings
import glob as _glob
import numpy as np
import pandas as pd
import matplotlib
try:
    get_ipython()
except NameError:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import Adam
import torchvision.models as tv_models
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    roc_curve, precision_recall_curve,
)
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family'      : 'DejaVu Sans',
    'font.size'        : 12,
    'axes.titlesize'   : 14,
    'axes.titleweight' : 'bold',
    'axes.labelsize'   : 12,
    'xtick.labelsize'  : 10,
    'ytick.labelsize'  : 10,
    'legend.fontsize'  : 10,
    'legend.framealpha': 0.9,
    'figure.dpi'       : 150,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.grid'        : True,
    'grid.alpha'       : 0.3,
    'grid.linestyle'   : '--',
})

PAL = {   # per-method colours (populated as methods are added)
    'ae': '#4878CF', 'aeu': '#F5A623', 'ae_pl': '#7B68EE',
    'dae': '#2ECC71', 'vae': '#95A5A6', 'ganomaly': '#E84C3D',
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {device}")
if device.type == 'cuda':
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    torch.backends.cudnn.benchmark = True





---
## Configuration
Hyperparameters follow the MedIAnomaly reference implementation exactly
(see CELL 1.4). We reproduce before we tune: any gap to their Table 6 must be
attributable to our code, not to a different operating point.

In [ ]:
# UNet + its GroupNorm/Swish/weight-standardised-conv building blocks, ported from
# MedIAnomaly (reconstruction/networks/). This is the architecture their DAE uses --
# the best pixel-level method in their benchmark (~20% Dice over 2nd place).
# Kept inline deliberately: a separate package needed a build step whose Makefile
# prerequisites silently broke on the space in 'Learn project', producing stale
# notebooks. One file = one source of truth.
from math import sqrt

# ---- inlined from networks/base_units/swish.py ------------------------------
# medium.com/the-artificial-impostor/more-memory-efficient-swish-activation-function-e07c22c12a76

class Swish(torch.autograd.Function):
    @staticmethod
    def forward(ctx, i):
        result = i * torch.sigmoid(i)
        ctx.save_for_backward(i)
        return result

    @staticmethod
    def backward(ctx, grad_output):
        i = ctx.saved_variables[0]
        sigmoid_i = torch.sigmoid(i)
        return grad_output * (sigmoid_i * (1 + i * (1 - sigmoid_i)))


class CustomSwish(nn.Module):
    def forward(self, input_tensor):
        return Swish.apply(input_tensor)


# ---- inlined from networks/base_units/ws_conv.py ------------------------------
from torch.nn import functional as F


# From https://github.com/joe-siyuan-qiao/WeightStandardization
class WNConv2d(nn.Conv2d):

    def __init__(self, in_channels, out_channels, kernel_size, stride=1,
                 padding=0, dilation=1, groups=1, bias=True):
        super(WNConv2d, self).__init__(in_channels, out_channels, kernel_size, stride,
                                       padding, dilation, groups, bias)

    def forward(self, x):
        weight = self.weight
        weight_mean = weight.mean(dim=1, keepdim=True).mean(dim=2,
                                                            keepdim=True).mean(dim=3, keepdim=True)
        weight = weight - weight_mean
        std = weight.view(weight.size(0), -1).std(dim=1).view(-1, 1, 1, 1) + 1e-5
        weight = weight / std.expand_as(weight)
        return F.conv2d(x, weight, self.bias, self.stride,
                        self.padding, self.dilation, self.groups)


# ---- inlined from networks/unet.py ------------------------------
from math import sqrt



def get_groups(channels: int) -> int:
    """
    :param channels:
    :return: return a suitable parameter for number of groups in GroupNormalisation'.
    """
    divisors = []
    for i in range(1, int(sqrt(channels)) + 1):
        if channels % i == 0:
            divisors.append(i)
            other = channels // i
            if i != other:
                divisors.append(other)
    return sorted(divisors)[len(divisors) // 2]


class UNet(nn.Module):
    def __init__(
            self,
            in_channels=1,
            n_classes=2,
            depth=5,
            wf=6,
            padding=True,
            norm="group",
            up_mode='upconv'):
        """
        A modified U-Net implementation [1].

        [1] U-Net: Convolutional Networks for Biomedical Image Segmentation
            Ronneberger et al., 2015 https://arxiv.org/abs/1505.04597

        Args:
            in_channels (int): number of input channels
            n_classes (int): number of output channels
            depth (int): depth of the network
            wf (int): number of filters in the first layer is 2**wf
            padding (bool): if True, apply padding such that the input shape
                            is the same as the output.
            norm (str): one of 'batch' and 'group'.
                        'batch' will use BatchNormalization.
                        'group' will use GroupNormalization.
            up_mode (str): one of 'upconv' or 'upsample'.
                           'upconv' will use transposed convolutions for learned upsampling.
                           'upsample' will use bilinear upsampling.
        """
        super(UNet, self).__init__()
        assert up_mode in ('upconv', 'upsample')
        self.padding = padding
        self.depth = depth
        prev_channels = in_channels
        self.down_path = nn.ModuleList()
        for i in range(depth):
            self.down_path.append(
                UNetConvBlock(prev_channels, 2 ** (wf + i), padding, norm=norm)
            )
            prev_channels = 2 ** (wf + i)

        self.up_path = nn.ModuleList()
        for i in reversed(range(depth - 1)):
            self.up_path.append(
                UNetUpBlock(prev_channels, 2 ** (wf + i), up_mode, padding, norm=norm)
            )
            prev_channels = 2 ** (wf + i)

        self.last = nn.Conv2d(prev_channels, n_classes, kernel_size=1)

    def forward_down(self, x):

        blocks = []
        for i, down in enumerate(self.down_path):
            x = down(x)
            blocks.append(x)
            if i != len(self.down_path) - 1:
                x = F.avg_pool2d(x, 2)

        return x, blocks

    def forward_up_without_last(self, x, blocks):
        for i, up in enumerate(self.up_path):
            skip = blocks[-i - 2]
            x = up(x, skip)

        return x

    def forward_without_last(self, x):
        x, blocks = self.forward_down(x)
        x = self.forward_up_without_last(x, blocks)
        return x

    def forward(self, x):
        x = self.get_features(x)
        # return self.last(x)
        return {'x_hat': self.last(x)}

    def get_features(self, x):
        return self.forward_without_last(x)


class UNetConvBlock(nn.Module):
    def __init__(self, in_size, out_size, padding, norm="group", kernel_size=3):
        super(UNetConvBlock, self).__init__()
        block = []
        if padding:
            block.append(nn.ReflectionPad2d(1))

        block.append(WNConv2d(in_size, out_size, kernel_size=kernel_size))
        block.append(CustomSwish())

        if norm == "batch":
            block.append(nn.BatchNorm2d(out_size))
        elif norm == "group":
            block.append(nn.GroupNorm(get_groups(out_size), out_size))

        if padding:
            block.append(nn.ReflectionPad2d(1))

        block.append(WNConv2d(out_size, out_size, kernel_size=kernel_size))
        block.append(CustomSwish())

        if norm == "batch":
            block.append(nn.BatchNorm2d(out_size))
        elif norm == "group":
            block.append(nn.GroupNorm(get_groups(out_size), out_size))

        self.block = nn.Sequential(*block)

    def forward(self, x):
        out = self.block(x)
        return out


class UNetUpBlock(nn.Module):
    def __init__(self, in_size, out_size, up_mode, padding, norm="group"):
        super(UNetUpBlock, self).__init__()
        if up_mode == 'upconv':
            self.up = nn.ConvTranspose2d(in_size, out_size, kernel_size=2, stride=2)
        elif up_mode == 'upsample':
            self.up = nn.Sequential(
                nn.Upsample(mode='bilinear', scale_factor=2),
                nn.Conv2d(in_size, out_size, kernel_size=1),
            )

        self.conv_block = UNetConvBlock(in_size, out_size, padding, norm=norm)

    def center_crop(self, layer, target_size):
        _, _, layer_height, layer_width = layer.size()
        diff_y = (layer_height - target_size[0]) // 2
        diff_x = (layer_width - target_size[1]) // 2
        return layer[:, :, diff_y: (diff_y + target_size[0]), diff_x: (diff_x + target_size[1])]

    def forward(self, x, bridge):
        up = self.up(x)
        crop1 = self.center_crop(bridge, up.shape[2:])
        out = torch.cat([up, crop1], 1)
        out = self.conv_block(out)

        return out


if __name__ == '__main__':
    model = UNet()



# ---- MedIAnomaly reconstruction backbone -------------------------------------
# AE / AE-U and their building blocks, ported UNCHANGED so our reproduction of
# Table 6 is not confounded by an architecture difference. AE-U subclasses AE and
# splits the final layer into (x_hat, log_var) for uncertainty-weighted scoring.


# ---- ported verbatim from MedIAnomaly/reconstruction/networks/base_units/conv_layers.py --------
def down_conv(in_planes, out_planes):
    return nn.Conv2d(in_planes, out_planes, kernel_size=4, stride=2, padding=1, bias=False)


def up_conv(in_planes, out_planes):
    return nn.ConvTranspose2d(in_planes, out_planes, kernel_size=4, stride=2, padding=1, bias=False)


def conv3x3(in_planes: int, out_planes: int, stride: int = 1, groups: int = 1, dilation: int = 1) -> nn.Conv2d:
    """3x3 convolution with padding"""
    return nn.Conv2d(
        in_planes,
        out_planes,
        kernel_size=3,
        stride=stride,
        padding=dilation,
        groups=groups,
        bias=False,
        dilation=dilation,
    )

# ---- ported verbatim from MedIAnomaly/reconstruction/networks/base_units/blocks.py --------------------
class BasicBlock(nn.Module):
    def __init__(self, inplanes, planes, num_layers, downsample=False, upsample=False, last_layer=False):
        super(BasicBlock, self).__init__()
        assert not (downsample and upsample)
        layers = []
        if downsample:
            layers.append(down_conv(inplanes, planes))
        elif upsample:
            layers.append(up_conv(inplanes, planes))
        else:
            layers.append(conv3x3(inplanes, planes))
        layers.append(nn.BatchNorm2d(planes))
        layers.append(nn.ReLU(inplace=True))

        # Deeper block
        if upsample:
            for _ in range(1, num_layers):
                add_layer = [conv3x3(inplanes, inplanes),
                             nn.BatchNorm2d(inplanes),
                             nn.ReLU(inplace=True)]

                layers = add_layer + layers
        else:
            for _ in range(1, num_layers):
                add_layer = [conv3x3(planes, planes),
                             nn.BatchNorm2d(planes),
                             nn.ReLU(inplace=True)]

                layers = layers + add_layer

        if last_layer:
            layers = layers[:-2]  # remove the BN and ReLU for the output layer.

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        out = self.model(x)
        return out


class ResBlock(nn.Module):
    def __init__(self, inplanes, planes, num_layers, downsample=False, upsample=False, last_layer=False):
        super(ResBlock, self).__init__()
        assert not (downsample and upsample)
        self.last_layer = last_layer
        self.relu = nn.ReLU(inplace=True)

        layers = []
        if downsample:
            layers.append(down_conv(inplanes, planes))
            self.skip = nn.Sequential(
                down_conv(inplanes, planes),
                nn.BatchNorm2d(planes)
            )
        elif upsample:
            layers.append(up_conv(inplanes, planes))
            self.skip = nn.Sequential(
                up_conv(inplanes, planes),
                nn.BatchNorm2d(planes)
            )
        else:
            layers.append(conv3x3(inplanes, planes))
            self.skip = nn.Identity()
        layers.append(nn.BatchNorm2d(planes))

        # Deeper block
        if upsample:
            for _ in range(1, num_layers):
                add_layer = [conv3x3(inplanes, inplanes),
                             nn.BatchNorm2d(inplanes),
                             nn.ReLU(inplace=True)]

                layers = add_layer + layers
        else:
            for _ in range(1, num_layers):
                add_layer = [nn.ReLU(inplace=True),
                             conv3x3(planes, planes),
                             nn.BatchNorm2d(planes)]

                layers = layers + add_layer

        if last_layer:  # remove the BN for the output layer.
            layers = layers[:-1]

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        identity = x

        out = self.model(x)

        identity = self.skip(identity)
        out += identity

        if not self.last_layer:  # remove the relu for the output layer.
            out = self.relu(out)

        return out


class BottleNeck(nn.Module):
    def __init__(self, in_planes, feature_size, mid_num=2048, latent_size=16):
        super(BottleNeck, self).__init__()
        self.in_planes = in_planes
        self.feature_size = feature_size
        self.linear_enc = nn.Sequential(
            nn.Linear(in_planes * feature_size * feature_size, mid_num),
            nn.BatchNorm1d(mid_num),
            nn.ReLU(True),
            nn.Linear(mid_num, latent_size))

        self.linear_dec = nn.Sequential(
            nn.Linear(latent_size, mid_num),
            nn.BatchNorm1d(mid_num),
            nn.ReLU(True),
            nn.Linear(mid_num, in_planes * feature_size * feature_size))

    def forward(self, x):
        x = x.view(x.size(0), -1)
        z = self.linear_enc(x)
        out = self.linear_dec(z)

        out = out.view(x.size(0), self.in_planes, self.feature_size, self.feature_size)

        return {'out': out, 'z': z}


class SpatialBottleNeck(nn.Module):
    def __init__(self, in_planes, feature_size, mid_num=2048, latent_size=16):
        super(SpatialBottleNeck, self).__init__()
        self.in_planes = in_planes
        self.feature_size = feature_size
        self.linear_enc = nn.Sequential(
            nn.Conv2d(in_channels=in_planes, out_channels=mid_num, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(mid_num),
            nn.ReLU(True),
            nn.Conv2d(in_channels=mid_num, out_channels=latent_size, kernel_size=1, stride=1, padding=0, bias=False))

        self.linear_dec = nn.Sequential(
            nn.Conv2d(in_channels=latent_size, out_channels=mid_num, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(mid_num),
            nn.ReLU(True),
            nn.Conv2d(in_channels=mid_num, out_channels=in_planes, kernel_size=1, stride=1, padding=0, bias=False),)

    def forward(self, x):
        z = self.linear_enc(x)
        out = self.linear_dec(z)

        return {'out': out, 'z': z}


class MemBottleNeck(BottleNeck):
    def __init__(self, in_planes, feature_size, mid_num=2048, latent_size=16, mem_size=25, shrink_thres=0.0025):
        super(MemBottleNeck, self).__init__(in_planes, feature_size, mid_num, latent_size)
        self.memory_module = MemModule(mem_dim=mem_size, fea_dim=latent_size, shrink_thres=shrink_thres)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        z = self.linear_enc(x)

        mem_out = self.memory_module(z)
        z_hat, att = mem_out['output'], mem_out['att']

        out = self.linear_dec(z_hat)
        out = out.view(x.size(0), self.in_planes, self.feature_size, self.feature_size)

        return {'out': out, 'att': att, 'z': z, 'z_hat': z_hat}


class VaeBottleNeck(BottleNeck):
    def __init__(self, in_planes, feature_size, mid_num=2048, latent_size=16):
        super(VaeBottleNeck, self).__init__(in_planes, feature_size, mid_num, latent_size)
        self.linear_enc = nn.Sequential(
            nn.Linear(in_planes * feature_size * feature_size, mid_num),
            nn.BatchNorm1d(mid_num),
            nn.ReLU(True),
            nn.Linear(mid_num, 2 * latent_size))

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return eps * std + mu

    def forward(self, x):
        x = x.view(x.size(0), -1)
        z = self.linear_enc(x)

        mu, log_var = z.chunk(2, dim=1)
        z_hat = self.reparameterize(mu, log_var)

        out = self.linear_dec(z_hat)
        out = out.view(x.size(0), self.in_planes, self.feature_size, self.feature_size)
        return {'out': out, 'mu': mu, 'log_var': log_var}

# ---- ported verbatim from MedIAnomaly/reconstruction/networks/ae.py --------------------
class AE(nn.Module):
    def __init__(self, input_size=64, in_planes=1, base_width=16, expansion=1, mid_num=2048, latent_size=16,
                 en_num_layers=1, de_num_layers=1, spatial=False):
        super(AE, self).__init__()

        bottleneck = SpatialBottleNeck if spatial else BottleNeck

        self.fm = input_size // 16  # down-sample for 4 times. 2^4=16

        self.en_block1 = BasicBlock(in_planes, 1 * base_width * expansion, en_num_layers, downsample=True)

        self.en_block2 = BasicBlock(1 * base_width * expansion, 2 * base_width * expansion, en_num_layers,
                                    downsample=True)
        self.en_block3 = BasicBlock(2 * base_width * expansion, 4 * base_width * expansion, en_num_layers,
                                    downsample=True)
        self.en_block4 = BasicBlock(4 * base_width * expansion, 4 * base_width * expansion, en_num_layers,
                                    downsample=True)

        self.bottle_neck = bottleneck(4 * base_width * expansion, feature_size=self.fm, mid_num=mid_num,
                                      latent_size=latent_size)

        self.de_block1 = BasicBlock(4 * base_width * expansion, 4 * base_width * expansion, de_num_layers,
                                    upsample=True)
        self.de_block2 = BasicBlock(4 * base_width * expansion, 2 * base_width * expansion, de_num_layers,
                                    upsample=True)
        self.de_block3 = BasicBlock(2 * base_width * expansion, 1 * base_width * expansion, de_num_layers,
                                    upsample=True)
        self.de_block4 = BasicBlock(1 * base_width * expansion, in_planes, de_num_layers, upsample=True,
                                    last_layer=True)

    def forward(self, x):
        en1 = self.en_block1(x)
        en2 = self.en_block2(en1)
        en3 = self.en_block3(en2)
        en4 = self.en_block4(en3)

        bottle_out = self.bottle_neck(en4)
        z, de4 = bottle_out['z'], bottle_out['out']

        de3 = self.de_block1(de4)
        de2 = self.de_block2(de3)
        de1 = self.de_block3(de2)
        x_hat = self.de_block4(de1)

        return {'x_hat': x_hat, 'z': z, 'en_features': [en1, en2, en3], 'de_features': [de1, de2, de3]}

# ---- ported verbatim from MedIAnomaly/reconstruction/networks/aeu.py --------------------
class AEU(AE):
    def __init__(self, input_size=64, in_planes=1, base_width=16, expansion=1, mid_num=2048, latent_size=16,
                 en_num_layers=None, de_num_layers=None):
        super(AEU, self).__init__(input_size, in_planes, base_width, expansion, mid_num, latent_size, en_num_layers,
                                  de_num_layers)

        self.de_block4 = BasicBlock(1 * base_width * expansion,  2 * in_planes, de_num_layers, upsample=True,
                                    last_layer=True)

    def forward(self, x):
        en1 = self.en_block1(x)
        en2 = self.en_block2(en1)
        en3 = self.en_block3(en2)
        en4 = self.en_block4(en3)

        bottle_out = self.bottle_neck(en4)
        z, de4 = bottle_out['z'], bottle_out['out']

        de3 = self.de_block1(de4)
        de2 = self.de_block2(de3)
        de1 = self.de_block3(de2)
        x_hat, log_var = self.de_block4(de1).chunk(2, 1)

        return {'x_hat': x_hat, 'log_var': log_var, 'z': z,
                'en_features': [en1, en2, en3], 'de_features': [de1, de2, de3]}

In [ ]:
# All values below are taken from the benchmark's own defaults
# (MedIAnomaly/reconstruction/options.py + data_utils.get_transform), NOT tuned by us.
# Rationale: our first goal is to REPRODUCE their numbers so ours are comparable to
# Table 6. Hyperparameter search (manual ablation / Optuna) comes later, on top of a
# verified baseline -- tuning before reproducing makes any gap uninterpretable.
SAMPLE_MODE = bool(int(os.environ.get('SAMPLE_MODE', '0')))

# Which dataset this run of the notebook is for. Everything below namespaces itself by
# this, so two datasets can never share a run id, a checkpoint directory or a wandb
# artifact name. This mirrors how the benchmark itself does it — MedIAnomaly puts the
# dataset at the TOP of its output path (options.py:68,
# result_dir = ~/Experiment/MedIAnomaly/{dataset}, then {model}/fold_{fold}) rather
# than branching its code per dataset.
DATASET = os.environ.get('DATASET', 'RSNA')

# Epochs are a property of the DATASET in the reference implementation
# (options.py:23 self.epochs), not a global. Brain Tumor is the odd one out at 600.
DATASET_EPOCHS = {'RSNA': 250, 'VinCXR': 250, 'LAG': 250,
                  'BrainTumor': 600, 'BraTS2021': 250}
assert DATASET in DATASET_EPOCHS, f'unknown DATASET {DATASET!r}'

# A NEW VERSION NAMESPACE, DELIBERATELY DISJOINT FROM dl-v1. RUN_VERSION names the
# checkpoint directory and the wandb artifact, so these 128px runs land in
# results_dl/ckpt_res128-v1-<dataset>/ and cannot collide with, overwrite or be confused
# for the 64px runs the paper reports. Nothing here touches dl-v1.
RUN_VERSION    = f'res128-v1-{DATASET.lower()}'
SKIP_COMPLETED = True
WANDB_PROJECT  = 'MedIAnomaly-DL'
WANDB_GROUP    = f'{RUN_VERSION}'

OUTPUT_DIR = ('/kaggle/working/results_dl' if os.path.isdir('/kaggle/working')
              else 'results_dl') + ('' if not SAMPLE_MODE else '_sample')
CKPT_DIR   = f'{OUTPUT_DIR}/ckpt_{RUN_VERSION}'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CKPT_DIR,   exist_ok=True)

# ── model / optimisation: MedIAnomaly options.py defaults ───────────────────────
IMAGE_SIZE    = 128         # THE POINT OF THIS NOTEBOOK. The benchmark claims 64 ~= 128;
                            # Limitation 1 of the paper predicts the SSIM rescore does not
                            # survive the change, because its window is a fixed 11x11 and
                            # therefore covers a quarter of the relative area at 128px.
LATENT_DIM    = 16          # options.py --latent-size
HIDDEN_NUM    = 1024        # options.py --hidden-num (bottleneck FC width)
BASE_WIDTH    = 16          # options.py --base-width
EN_DEPTH      = 1           # options.py --en-depth
DE_DEPTH      = 1           # options.py --de-depth

# DAE's UNet is a separate architecture with its own defaults (networks/unet.py), not a
# variant of the AE above. See the note in build_net about Table 6's params column.
DAE_UNET_DEPTH = 5
DAE_UNET_WF    = 6          # first layer has 2**wf channels
EPOCHS        = DATASET_EPOCHS[DATASET] if not SAMPLE_MODE else 2   # options.py epochs[...]
BATCH_SIZE    = 64  if not SAMPLE_MODE else 4      # options.py --train-batch-size
LR            = 1e-3                                # options.py --train-lr
WEIGHT_DECAY  = 0.0                                 # options.py --train-weight-decay
EPS           = 1e-8

# Images are normalised to [-1, 1], matching data_utils.get_transform:
#   transforms.Normalize((0.5,), (0.5,))  applied after ToTensor().
# This matters: their SSIM loss does ((x+1)/2) internally, and any loss/noise ported
# from their code assumes this range. Do NOT silently switch to [0, 1].
PIXEL_RANGE = (-1.0, 1.0)

# ── seeds ───────────────────────────────────────────────────────────────────────
# The train/test split now comes from data.json (their split), so SPLIT_SEED no longer
# controls it -- only model init / batch order / augmentation.
TRAIN_SEED = int(os.environ.get('TRAIN_SEED', '42'))
SEED       = TRAIN_SEED
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"DATASET     : {DATASET}")
print(f"SAMPLE_MODE : {SAMPLE_MODE}")
print(f"RUN_VERSION : {RUN_VERSION}  (SKIP_COMPLETED={SKIP_COMPLETED})")
print(f"image {IMAGE_SIZE}px | latent {LATENT_DIM} | epochs {EPOCHS} | bs {BATCH_SIZE} | lr {LR}")
print(f"OUTPUT_DIR  : {OUTPUT_DIR}")

# Name of the Kaggle Secret holding your wandb API key.
# Kaggle: Notebook -> Add-ons -> Secrets -> add a secret with THIS label.
# Locally: set WANDB_API_KEY, or just run `wandb login` once.
WANDB_SECRET_NAME = os.environ.get('WANDB_SECRET_NAME', 'REATTN_KEY')

USE_WANDB = False

# C1.5 Checkpoint save/load helpers

In [ ]:
# Replaces the previous save_ckpt/load_ckpt, which were built for a different study and
# caused two real bugs: a positional `disc_scores` slot that later got reused to smuggle
# pixel maps, and artifact names that save/restore built differently. This design makes
# both mistakes structurally impossible:
#   * every payload is NAMED (arrays={'scores':..., 'pixel_maps':...}) — no positional slots
#   * one deterministic run_id derived from (method, params, seed) — no hand-built names
#   * a MANIFEST records the global config fingerprint; loading verifies it and REFUSES
#     to hand back a record produced under different settings (that is the parameter-mixing
#     guard: a result trained at 64px/lr1e-3 can never silently be reused at 128px/lr1e-4)

import hashlib, datetime

all_results  = {}    # run_id -> metrics dict
loss_history = {}    # run_id -> list of per-epoch losses

# Global settings that MUST match for a stored run to be reusable. Anything that changes
# the numbers belongs here; anything cosmetic must not (or every run invalidates).
def config_fingerprint():
    return {
        'image_size': IMAGE_SIZE, 'latent_dim': LATENT_DIM, 'hidden_num': HIDDEN_NUM,
        'base_width': BASE_WIDTH, 'en_depth': EN_DEPTH, 'de_depth': DE_DEPTH,
        'dae_unet_depth': DAE_UNET_DEPTH, 'dae_unet_wf': DAE_UNET_WF,
        'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'lr': LR,
        'weight_decay': WEIGHT_DECAY, 'pixel_range': list(PIXEL_RANGE),
        'sample_mode': SAMPLE_MODE, 'run_version': RUN_VERSION,
        'dataset': DATASET,
    }


# Runs stored before DATASET existed have no 'dataset' key in their manifest, and every
# one of them was RSNA. Defaulting the MISSING key at comparison time (never by rewriting
# a stored manifest) keeps those runs valid while still making a genuine cross-dataset
# mix-up raise.
_LEGACY_FINGERPRINT_DEFAULTS = {'dataset': 'RSNA'}


def _slug(v):
    """Compact, filesystem- and wandb-safe rendering of a parameter value."""
    if isinstance(v, float):
        return f'{v:g}'.replace('.', 'p').replace('-', 'm')
    return str(v).replace('.', 'p').replace('/', '-').replace(' ', '')


def run_id(method, seed, **params):
    """Deterministic id: method[_ds-X][_k-v...]_sN. Same inputs -> same id, always.

    The dataset tag is OMITTED for RSNA on purpose. Every RSNA run already stored locally
    and on wandb was written under the un-tagged id, and adding a tag unconditionally
    would orphan all of them. RSNA is the historical default; every other dataset is
    tagged, so a collision across datasets is impossible in either direction."""
    parts = [method]
    if DATASET != 'RSNA':
        parts.append(f'ds-{_slug(DATASET)}')
    parts += [f'{k}-{_slug(v)}' for k, v in sorted(params.items())] + [f's{seed}']
    rid = '_'.join(parts)
    return rid if len(rid) <= 100 else f'{method}_{hashlib.md5(rid.encode()).hexdigest()[:12]}_s{seed}'


def _run_dir(rid):
    return os.path.join(CKPT_DIR, rid)


def _manifest_path(rid):
    return os.path.join(_run_dir(rid), 'manifest.json')


def run_exists(rid):
    return SKIP_COMPLETED and os.path.isfile(_manifest_path(rid))


def save_run(rid, *, method, seed, params, metrics, epoch_loss=None,
             arrays=None, weights=None, extra=None):
    """Persist one run: manifest + named arrays + named weights, then upload as ONE
    wandb artifact. Arrays and weights are written in the same call, so a record can
    never contain arrays from one model and weights from another."""
    d = _run_dir(rid); os.makedirs(d, exist_ok=True)
    files = []
    for name, arr in (arrays or {}).items():
        fp = os.path.join(d, f'{name}.npy'); np.save(fp, np.asarray(arr)); files.append(f'{name}.npy')
    for name, sd in (weights or {}).items():
        fp = os.path.join(d, f'{name}.pth'); torch.save(sd, fp); files.append(f'{name}.pth')

    manifest = {
        'run_id': rid, 'method': method, 'seed': seed, 'params': params,
        'metrics': {k: (float(v) if isinstance(v, (int, float, np.floating)) else v)
                    for k, v in metrics.items()},
        'epoch_loss': [float(v) for v in (epoch_loss or [])],
        'config': config_fingerprint(),
        'arrays': list((arrays or {}).keys()), 'weights': list((weights or {}).keys()),
        'files': files, 'saved_at': datetime.datetime.now().isoformat(timespec='seconds'),
        'extra': extra or {},
    }
    with open(_manifest_path(rid), 'w') as f:
        json.dump(manifest, f, indent=2)

    all_results[rid] = manifest['metrics']
    if epoch_loss:
        loss_history[rid] = manifest['epoch_loss']

    if USE_WANDB and wandb.run is not None:
        wandb.log({f'{method}/{k}': v for k, v in manifest['metrics'].items()
                   if isinstance(v, (int, float))} | {'run_id': rid})
        try:
            art = wandb.Artifact(f'{WANDB_GROUP}-{rid}'.lower().replace('.', 'p'),
                                 type='run', metadata=manifest)
            art.add_dir(d)
            art = wandb.log_artifact(art); art.wait()
            print(f'  [{rid}] artifact -> {art.name}')
        except Exception as e:
            print(f'  [{rid}] artifact upload failed: {e}')
    print(f'  [{rid}] saved ({len(files)} files) -> {d}')
    return manifest


def load_run(rid, models=None, strict=True):
    """Load a stored run. Returns (manifest, arrays_dict). Verifies the stored config
    fingerprint against the CURRENT one and refuses on mismatch when strict — this is
    what stops a run trained under different hyperparameters being silently reused."""
    with open(_manifest_path(rid)) as f:
        man = json.load(f)
    cur, old = config_fingerprint(), man.get('config', {})
    _d = _LEGACY_FINGERPRINT_DEFAULTS
    diff = {k: (old.get(k, _d.get(k)), cur[k])
            for k in cur if old.get(k, _d.get(k)) != cur[k]}
    if diff:
        msg = (f"[{rid}] CONFIG MISMATCH — stored run used different settings:\n" +
               '\n'.join(f'    {k}: stored={a!r} current={b!r}' for k, (a, b) in diff.items()))
        if strict:
            raise RuntimeError(msg + '\n  Delete the run dir to retrain, or set strict=False '
                                     'if you deliberately want to mix.')
        print('  WARNING ' + msg)

    d = _run_dir(rid)
    arrays = {n: np.load(os.path.join(d, f'{n}.npy')) for n in man['arrays']}
    for name, model in (models or {}).items():
        fp = os.path.join(d, f'{name}.pth')
        if not os.path.isfile(fp):
            raise FileNotFoundError(f"[{rid}] weights {name!r} missing at {fp}")
        model.load_state_dict(torch.load(fp, map_location=device))
        model.eval()          # reload path is always inference; train-mode BatchNorm
                              # would use batch stats AND mutate running stats
    all_results[rid] = man['metrics']
    if man.get('epoch_loss'):
        loss_history[rid] = man['epoch_loss']
    print(f"  [{rid}] loaded (saved {man['saved_at']})")
    return man, arrays


def fetch_run(rid, run_version=None, entity=None):
    """Pull a run's artifact from wandb into CKPT_DIR if it is not already local."""
    if os.path.isfile(_manifest_path(rid)):
        return True
    try:
        api = wandb.Api()
        name = f'{run_version or RUN_VERSION}-{rid}'.lower().replace('.', 'p')
        art = api.artifact(f'{entity or api.default_entity}/{WANDB_PROJECT}/{name}:latest')
        art.download(root=_run_dir(rid))
        print(f'  [{rid}] restored from wandb')
        return True
    except Exception as e:
        # Print the MESSAGE, not just the class. wandb raises CommError both for
        # 'artifact does not exist yet' (benign, the normal first-run path) and for a
        # genuine network/auth failure (not benign — it silently retrains everything on
        # a fresh session). Those two are indistinguishable from the class name alone.
        msg = str(e).replace('\n', ' ')[:200]
        benign = 'not found' in msg.lower() or 'does not exist' in msg.lower()
        tag = 'absent' if benign else 'FETCH FAILED'
        print(f'  [{rid}] {tag} ({type(e).__name__}: {msg}) — will train fresh')
        return False


def completed_runs():
    """Every run stored under the current CKPT_DIR, as a DataFrame."""
    rows = []
    for mp in sorted(_glob.glob(os.path.join(CKPT_DIR, '*', 'manifest.json'))):
        with open(mp) as f:
            m = json.load(f)
        rows.append({'run_id': m['run_id'], 'method': m['method'], 'seed': m['seed'],
                     **m['params'], **m['metrics']})
    return pd.DataFrame(rows)

# C1.6 Wandb setup and login

In [ ]:
# USE_WANDB is the flag every later cell should check before calling wandb.*
# — that's what makes wandb fully optional (see save_ckpt above).
try:
    import wandb
    # Three login paths, tried in order: explicit env var -> Kaggle Secrets -> netrc.
    # Each failure is reported SPECIFICALLY: a missing secret, a wrong secret NAME and
    # a rejected key all used to look identical ("wandb unavailable"), which made this
    # impossible to debug from the notebook output.
    if os.environ.get('WANDB_API_KEY'):
        wandb.login(key=os.environ['WANDB_API_KEY'], relogin=True)
        print('WandB: logged in via WANDB_API_KEY')
    elif os.path.isdir('/kaggle/working'):
        from kaggle_secrets import UserSecretsClient
        try:
            _key = UserSecretsClient().get_secret(WANDB_SECRET_NAME)
        except Exception as _se:
            raise RuntimeError(
                f"Kaggle Secret {WANDB_SECRET_NAME!r} not readable ({_se}). "
                f"Add-ons -> Secrets: create a secret labelled {WANDB_SECRET_NAME!r} "
                f"AND tick 'attach to notebook', or set WANDB_SECRET_NAME to its label."
            ) from None
        if not _key:
            raise RuntimeError(f"Kaggle Secret {WANDB_SECRET_NAME!r} is empty.")
        wandb.login(key=_key, relogin=True)
        print(f'WandB: logged in via Kaggle Secret {WANDB_SECRET_NAME!r}')
    else:
        wandb.login()
        print('WandB: logged in via netrc / interactive')
    USE_WANDB = True
    # id=RUN_VERSION + resume='allow' means re-running this cell
    # (e.g. after a Kaggle session reset) reattaches to the SAME wandb run
    # instead of creating a new one, so metrics keep appending to one history.
    wandb.init(project=WANDB_PROJECT,
               group=WANDB_GROUP,
               name=f'{RUN_VERSION}',
               config=dict(image_size=IMAGE_SIZE, latent_dim=LATENT_DIM,
                           hidden_num=HIDDEN_NUM, base_width=BASE_WIDTH,
                           epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR,
                           weight_decay=WEIGHT_DECAY, pixel_range=PIXEL_RANGE,
                           run_version=RUN_VERSION),
               tags=['medianomaly', 'dl', RUN_VERSION],
               resume='allow', id=f'{RUN_VERSION}',
               settings=wandb.Settings(init_timeout=120))
    print(f'WandB ready  project={WANDB_PROJECT}  version={RUN_VERSION}')
except Exception as _e:
    # Any failure here (no internet, no key, user declines login, etc.)
    # falls back to USE_WANDB=False so the rest of the notebook still runs.
    USE_WANDB = False
    print(f'WandB unavailable ({_e}) — continuing without.')

# Datasets this project needs: image-level classification only -> RSNA is enough
# ONE dataset per notebook run. find_data_root returns the first root containing EVERY
# name in `required`, so asking for two at once fails outright when they live under
# different roots (a Kaggle mount and a local download, say) with an error that points at
# the wrong problem.
REQUIRED_DATASETS = [DATASET]

---
## **Cell 1.7** — Dataset-isolation self-test (run this before anything else)
The run-storage layer was built to stop *hyperparameter* mixing. It did not stop
**dataset** mixing: `run_id` had no dataset field and `config_fingerprint` had no
dataset key, so a second dataset's `ae_s42` would have been the same id, the same
directory and the same wandb artifact as RSNA's. `run_exists` would return True,
the fingerprint would show no difference, and `load_run` would hand back RSNA's
scores to be evaluated against the new dataset's labels.

That failure is **silent** for exactly the datasets we most want to run: VinCXR and LAG
both train for 250 epochs, so the fingerprint stays bit-identical to RSNA's, and
VinCXR's test set is also 2000 images, so `roc_auc_score` would not even raise on a
length mismatch. The result would be a complete, plausible, entirely fictitious table.

So this cell does not merely *apply* the fix — it **asserts** it. Two properties matter
and they pull in opposite directions:

1. **RSNA ids must be byte-identical to what is already stored**, or every artifact
   already on wandb is orphaned and the whole project silently retrains.
2. **Every other dataset must be unreachable from RSNA's namespace**, in both
   directions, and a cross-dataset load must RAISE rather than return.

In [ ]:

def dataset_isolation_selftest(verbose=True):
    """Asserts the dataset namespacing is correct. Cheap, and the only thing standing
    between a second dataset and a fictitious results table."""
    ok = True

    def check(label, cond):
        nonlocal ok
        ok &= bool(cond)
        if verbose:
            print(f"  {'PASS' if cond else 'FAIL'}  {label}")

    # --- 1. RSNA's historical ids and namespaces are untouched ----------------------
    if DATASET == 'RSNA':
        check("RSNA run id unchanged            run_id('ae',42) == 'ae_s42'",
              run_id('ae', 42) == 'ae_s42')
        check("RSNA param id unchanged          ...w-2_s42",
              run_id('ae-posthoc-u', 42, w='2') == 'ae-posthoc-u_w-2_s42')
        # INVERTED FOR THIS NOTEBOOK. In dl_project.py these two assert that RSNA still
        # writes under 'dl-v1'. Here the whole point is that it must NOT: this study
        # trains at a different resolution, so its runs have to land in a namespace of
        # their own or they would silently overwrite the 64px results the paper reports.
        check("res128 namespace is NOT dl-v1", RUN_VERSION != 'dl-v1'
              and not RUN_VERSION.startswith('dl-v1'))
        check("res128 RUN_VERSION is the expected one",
              RUN_VERSION == f'res128-v1-{DATASET.lower()}')
        check("res128 wandb artifact cannot collide with dl-v1",
              not f'{WANDB_GROUP}-{run_id("ae", 42)}'.lower().startswith('dl-v1'))
        check("checkpoint dir is disjoint from the 64px runs",
              'res128' in CKPT_DIR and 'ckpt_dl-v1' not in CKPT_DIR)
        # a legacy manifest (no 'dataset' key) must still validate under the shim
        cur = config_fingerprint()
        legacy = {k: v for k, v in cur.items() if k != 'dataset'}
        d = _LEGACY_FINGERPRINT_DEFAULTS
        check("legacy manifest (no dataset key) still validates",
              not {k for k in cur if legacy.get(k, d.get(k)) != cur[k]})
    else:
        check(f"{DATASET} id is tagged           {run_id('ae', 42)}",
              run_id('ae', 42) == f'ae_ds-{_slug(DATASET)}_s42')
        check(f"{DATASET} id differs from RSNA's", run_id('ae', 42) != 'ae_s42')
        check(f"{DATASET} RUN_VERSION namespaced {RUN_VERSION}",
              RUN_VERSION == f'dl-v1-{DATASET.lower()}')
        check(f"{DATASET} CKPT_DIR namespaced", RUN_VERSION in CKPT_DIR)
        # an RSNA manifest loaded under this dataset MUST raise, not return
        rsna_rid = 'ae_s42'
        legacy_path = os.path.join(OUTPUT_DIR, 'ckpt_dl-v1', rsna_rid, 'manifest.json')
        if os.path.isfile(legacy_path):
            with open(legacy_path) as f:
                stored = json.load(f).get('config', {})
            d = _LEGACY_FINGERPRINT_DEFAULTS
            cur = config_fingerprint()
            check("an RSNA-stored run is REJECTED under this dataset",
                  bool({k for k in cur if stored.get(k, d.get(k)) != cur[k]}))
        else:
            print('  SKIP  no local RSNA manifest to cross-check against')

    # --- 2. params always separate ids ---------------------------------------------
    check("head widths do not collide",
          run_id('ae-posthoc-u', 42, w='2') != run_id('ae-posthoc-u', 42, w='8'))
    check("an epochs override separates ids",
          run_id('ae', 42, ep=50) != run_id('ae', 42))

    # --- 3. the ensemble member resolver carries params -----------------------------
    # A3_MEMBERS is defined much later (Cell 5.2), so this part only runs when the
    # self-test is re-invoked after the whole notebook has been executed. Skipping it on
    # the first pass is correct, not a hole: nothing before Cell 5.2 can use it.
    if 'A3_MEMBERS' in globals():
        check("A3 members carry params (3-tuples)",
              all(len(m) == 3 for m in A3_MEMBERS) and len(A3_MEMBERS) == 4)
        check("the extended set adds the post-hoc head with w='2'",
              ('ssim+head', 'ae-ssim-posthoc-u', {'w': '2'}) in A3_MEMBERS_EXT)
    else:
        print('  SKIP  A3 member checks (A3_MEMBERS not defined until Cell 5.2)')

    print(f"\n  dataset-isolation self-test: {'PASS' if ok else 'FAIL'}   "
          f"(DATASET={DATASET}, RUN_VERSION={RUN_VERSION})")
    if not ok:
        raise RuntimeError('dataset isolation is NOT safe — do not run experiments')
    return ok


dataset_isolation_selftest()

# BEFORE the first run on a NEW dataset, back the checkpoint store up. Ten seconds, and
# it is the only thing that makes any of the above reversible:
#     cp -r results_dl/ckpt_dl-v1 results_dl/ckpt_dl-v1.bak

---
## **Cell 2.0** — Dataset acquisition (MedIAnomaly preprocessed data)
We use the benchmark's OWN preprocessed data rather than raw Kaggle DICOMs, because
`data.json` encodes their exact train/test split. That is what makes our numbers
directly comparable to MedIAnomaly Table 6 instead of "similar setup, different split".

Expected layout (from https://zenodo.org/records/12677223):
  <DATA_ROOT>/RSNA/{images/, data.json}        image-level  (3851 train / 1000+1000 test)
  <DATA_ROOT>/VinCXR/{images/, data.json}      image-level
  <DATA_ROOT>/BraTS2021/train/                 pixel-level  (the ONLY dataset with masks)
                       /test/{normal,tumor,annotation}/

The cell searches several roots so the same notebook works on Kaggle (dataset mounted
under /kaggle/input) and locally, and only downloads if nothing is found.

In [ ]:

import glob as _glob, tarfile, urllib.request

ZENODO = "https://zenodo.org/records/12677223/files/{}.tar.gz?download=1"

# Searched in order. Kaggle Datasets land in /kaggle/input/<slug>/ and the slug is
# user-chosen, so we glob for any directory that contains the expected dataset folders.
def _discover_roots(base='/kaggle/input', max_depth=4):
    """Bounded walk of the Kaggle mount, returning every directory that could be a root.

    Fixed glob patterns are not enough: Kaggle mounts inputs at DIFFERENT DEPTHS
    depending on how the notebook was created. The classic layout is
    /kaggle/input/<slug>/, but the namespaced one is
    /kaggle/input/datasets/<owner>/<slug>/ — two levels deeper, which a
    '/kaggle/input/*' glob never reaches. Rather than enumerate a pattern per layout,
    walk a few levels and let _looks_like decide which directory is real.

    Image folders are pruned: the RSNA competition input alone holds ~27k DICOMs, and
    descending into it would cost seconds for directories that can never be a root."""
    out = []
    if not os.path.isdir(base):
        return out
    base_depth = base.rstrip('/').count(os.sep)
    for root, dirs, _ in os.walk(base):
        out.append(root)
        if root.count(os.sep) - base_depth >= max_depth:
            dirs[:] = []
            continue
        dirs[:] = [d for d in dirs
                   if d not in ('images', 'train', 'test', 'annotation', 'normal', 'tumor')]
    return out


DATA_ROOT_CANDIDATES = [
    os.environ.get('MEDIANOMALY_DATA', ''),
    os.path.expanduser('~/MedIAnomaly-Data'),
    '/kaggle/working/MedIAnomaly-Data',
] + _discover_roots()


def _looks_like(root, name):
    """True if <root>/<name> has the structure the MedIAnomaly loaders expect."""
    d = os.path.join(root, name)
    if name == 'BraTS2021':
        return all(os.path.isdir(os.path.join(d, p))
                   for p in ['train', 'test/normal', 'test/tumor', 'test/annotation'])
    return os.path.isdir(os.path.join(d, 'images')) and os.path.isfile(os.path.join(d, 'data.json'))


def find_data_root(required):
    """Return the first candidate root that contains every dataset in `required`."""
    for root in DATA_ROOT_CANDIDATES:
        if root and os.path.isdir(root) and all(_looks_like(root, n) for n in required):
            return root
    return None


def download_datasets(names, root=None, force=False):
    """Fetch + extract from Zenodo. NOT called automatically — these archives are large
    and Kaggle sessions have limited disk, so downloading is an explicit decision."""
    root = root or os.path.expanduser('~/MedIAnomaly-Data')
    os.makedirs(root, exist_ok=True)
    for name in names:
        if _looks_like(root, name) and not force:
            print(f'  {name}: already present, skipping'); continue
        tgz = os.path.join(root, f'{name}.tar.gz')
        if not os.path.exists(tgz) or force:
            print(f'  {name}: downloading …')
            urllib.request.urlretrieve(ZENODO.format(name), tgz)
        print(f'  {name}: extracting …')
        with tarfile.open(tgz, 'r:gz') as t:
            t.extractall(root)
        os.remove(tgz)
        print(f'  {name}: {"OK" if _looks_like(root, name) else "STRUCTURE UNEXPECTED"}')
    return root


def verify_datasets(required, root=None):
    """Print a per-dataset report and RAISE if anything required is missing, so the
    notebook fails here with an actionable message instead of deep inside training."""
    root = root or find_data_root(required)
    print(f'DATA_ROOT: {root}')
    if root is None:
        print('  searched:', [c for c in DATA_ROOT_CANDIDATES if c])
        raise FileNotFoundError(
            'MedIAnomaly data not found. Either:\n'
            f'  (a) download_datasets({required})   # needs internet + disk\n'
            '  (b) upload the extracted MedIAnomaly-Data folder as a Kaggle Dataset, or\n'
            '  (c) set MEDIANOMALY_DATA=/path/to/MedIAnomaly-Data')
    ok = True
    for name in required:
        d = os.path.join(root, name)
        if not _looks_like(root, name):
            print(f'  {name:<12} MISSING or wrong structure at {d}'); ok = False; continue
        if name == 'BraTS2021':
            n_tr = len(os.listdir(os.path.join(d, 'train')))
            n_no = len(os.listdir(os.path.join(d, 'test/normal')))
            n_tu = len(os.listdir(os.path.join(d, 'test/tumor')))
            n_an = len(os.listdir(os.path.join(d, 'test/annotation')))
            print(f'  {name:<12} train={n_tr}  test normal={n_no}  tumor={n_tu}  masks={n_an}')
            if n_tu != n_an:
                print(f'    WARNING: {n_tu} tumor images but {n_an} masks — pixel metrics need a mask per image')
                ok = False
        else:
            with open(os.path.join(d, 'data.json')) as f:
                dd = json.load(f)
            n_img = len(os.listdir(os.path.join(d, 'images')))
            tr, te0, te1 = len(dd['train']['0']), len(dd['test']['0']), len(dd['test']['1'])
            print(f'  {name:<12} train={tr}  test normal={te0}  abnormal={te1}  files={n_img}')
            if name == 'RSNA' and (tr, te0, te1) != (3851, 1000, 1000):
                print(f'    WARNING: expected 3851/1000/1000 (MedIAnomaly Table 2), got {tr}/{te0}/{te1}')
    if not ok:
        raise RuntimeError('dataset verification failed — see report above')
    print('All required datasets verified.')
    return root


DATA_ROOT = verify_datasets(REQUIRED_DATASETS)

In [ ]:

def load_split_imagelevel(root, name, size=IMAGE_SIZE):
    """RSNA / VinCXR: returns (x_train, x_test, y_test) using THEIR split from data.json."""
    from PIL import Image
    d = os.path.join(root, name)
    with open(os.path.join(d, 'data.json')) as f:
        dd = json.load(f)

    def _load(names, tag):
        out = []
        for i, nm in enumerate(names):
            if i % 500 == 0:
                print(f'  {tag}: {i}/{len(names)}')
            im = Image.open(os.path.join(d, 'images', nm)).convert('L').resize((size, size),
                                                                               Image.BILINEAR)
            out.append(np.asarray(im, dtype=np.float32) / 127.5 - 1.0)   # -> [-1, 1]
        return np.stack(out)[:, None] if out else np.zeros((0, 1, size, size), np.float32)

    tr  = dd['train']['0']
    te0, te1 = dd['test']['0'], dd['test']['1']
    if SAMPLE_MODE:                      # smoke test: tiny subsets, same code path
        tr, te0, te1 = tr[:30], te0[:10], te1[:5]
    x_train = _load(tr,  f'{name}-train')
    x_test  = np.concatenate([_load(te0, f'{name}-test-normal'), _load(te1, f'{name}-test-abn')])
    y_test  = np.array([0] * len(te0) + [1] * len(te1), dtype=np.int32)
    print(f'{name}: train {x_train.shape}  test {x_test.shape}  ({y_test.mean()*100:.1f}% abnormal)')
    return x_train, x_test, y_test


def load_split_brats(root, size=IMAGE_SIZE):
    """BraTS2021: returns (x_train, x_test, y_test, masks). The ONLY dataset here with
    pixel-level ground truth — masks are 0/255 PNGs named like the image with
    'flair'->'seg', matching MedIAnomaly's BraTSAD loader."""
    from PIL import Image
    d = os.path.join(root, 'BraTS2021')

    def _load(dirpath, names, tag, nearest=False):
        out = []
        for i, nm in enumerate(names):
            if i % 500 == 0:
                print(f'  {tag}: {i}/{len(names)}')
            im = Image.open(os.path.join(dirpath, nm)).convert('L').resize(
                (size, size), Image.NEAREST if nearest else Image.BILINEAR)
            out.append(np.asarray(im, dtype=np.float32) / 127.5 - 1.0)   # -> [-1, 1]
        return np.stack(out)[:, None] if out else np.zeros((0, 1, size, size), np.float32)

    tr_names = sorted(os.listdir(os.path.join(d, 'train')))
    no_names = sorted(os.listdir(os.path.join(d, 'test/normal')))
    tu_names = sorted(os.listdir(os.path.join(d, 'test/tumor')))
    if SAMPLE_MODE:
        tr_names, no_names, tu_names = tr_names[:30], no_names[:10], tu_names[:5]
    mk_names = [e.replace('flair', 'seg') for e in tu_names]

    x_train = _load(os.path.join(d, 'train'), tr_names, 'brats-train')
    x_test  = np.concatenate([_load(os.path.join(d, 'test/normal'), no_names, 'brats-normal'),
                               _load(os.path.join(d, 'test/tumor'),  tu_names, 'brats-tumor')])
    y_test  = np.array([0] * len(no_names) + [1] * len(tu_names), dtype=np.int32)
    masks   = np.concatenate([np.zeros((len(no_names), 1, size, size), np.float32),
                               _load(os.path.join(d, 'test/annotation'), mk_names, 'brats-masks',
                                     nearest=True)])
    masks = (masks > 0.0).astype(np.float32)      # loader gives [-1,1]; >0 == was >127 == mask
    print(f'BraTS2021: train {x_train.shape}  test {x_test.shape}  '
          f'masks {masks.shape}  positive pixels {masks.mean()*100:.2f}%')
    return x_train, x_test, y_test, masks

---
## **Cell 2.2** — Load RSNA into memory
The whole dataset at 64px is ~30 MB, so we hold it as tensors and skip a `Dataset`
class entirely. Loading ONCE here (rather than per experiment) is deliberate: every
method below then sees byte-identical inputs, so a difference between two rows of the
results table can only come from the method, never from a re-decoded image.

In [ ]:

x_train_np, x_test_np, y_test_np = load_split_imagelevel(DATA_ROOT, DATASET)

X_TRAIN = torch.from_numpy(x_train_np)                 # (N,1,64,64) float32 in [-1,1]
X_TEST  = torch.from_numpy(x_test_np)
Y_TEST  = y_test_np

assert X_TRAIN.dtype == torch.float32 and X_TRAIN.shape[1] == 1
assert -1.01 <= float(X_TRAIN.min()) and float(X_TRAIN.max()) <= 1.01, \
    "inputs must be in [-1,1] — the ported SSIM/perceptual losses assume it"
print(f'X_TRAIN {tuple(X_TRAIN.shape)}  X_TEST {tuple(X_TEST.shape)}  '
      f'abnormal {Y_TEST.mean()*100:.1f}%  range [{X_TRAIN.min():.2f}, {X_TRAIN.max():.2f}]')

---
## **Cell 3.0** — Losses and anomaly scores
Ported from `MedIAnomaly/reconstruction/utils/losses.py` and `utils/util.py`.

Every loss shares one interface, which is the key design idea of their codebase and
the thing that makes experiment **A1** below possible at all:

```python
criterion(x, net_out)                                     -> scalar training loss
criterion(x, net_out, anomaly_score=True)                 -> (N,)      per-image score
criterion(x, net_out, anomaly_score=True, keepdim=True)   -> (N,1,H,W) per-pixel map
```

Because the training objective and the anomaly score are the *same* callable, they are
normally locked together — train with L2, score with L2. A1 breaks that link on purpose
and crosses them, which their paper never does.

All losses assume inputs in **[-1, 1]** (`SSIMLoss` rescales to [0,1] internally,
`PerceptualLoss._preprocess` de-normalises with mean=std=0.5).

In [ ]:

from typing import List, Optional, Tuple, Union
from collections import OrderedDict

# ---- SSIM, ported verbatim from MedIAnomaly/reconstruction/utils/util.py -------------
def _fspecial_gauss_1d(size: int, sigma: float) -> torch.Tensor:
    coords = torch.arange(size, dtype=torch.float)
    coords -= size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g /= g.sum()
    return g.unsqueeze(0).unsqueeze(0)


def gaussian_filter(input: torch.Tensor, win: torch.Tensor) -> torch.Tensor:
    assert all([ws == 1 for ws in win.shape[1:-1]]), win.shape
    if len(input.shape) == 4:
        conv = F.conv2d
    elif len(input.shape) == 5:
        conv = F.conv3d
    else:
        raise NotImplementedError(input.shape)
    C = input.shape[1]
    out = input
    for i, s in enumerate(input.shape[2:]):
        if s >= win.shape[-1]:
            out = conv(out, weight=win.transpose(2 + i, -1), stride=1, padding=0, groups=C)
        else:
            warnings.warn(f"Skipping Gaussian Smoothing at dim 2+{i} for {input.shape}")
    return out


def _ssim(X, Y, data_range, win, size_average=True, K=(0.01, 0.03)):
    K1, K2 = K
    compensation = 1.0
    C1, C2 = (K1 * data_range) ** 2, (K2 * data_range) ** 2
    win = win.to(X.device, dtype=X.dtype)

    mu1, mu2 = gaussian_filter(X, win), gaussian_filter(Y, win)
    mu1_sq, mu2_sq, mu1_mu2 = mu1.pow(2), mu2.pow(2), mu1 * mu2

    sigma1_sq = compensation * (gaussian_filter(X * X, win) - mu1_sq)
    sigma2_sq = compensation * (gaussian_filter(Y * Y, win) - mu2_sq)
    sigma12   = compensation * (gaussian_filter(X * Y, win) - mu1_mu2)

    cs_map   = (2 * sigma12 + C2) / (sigma1_sq + sigma2_sq + C2)
    ssim_map = ((2 * mu1_mu2 + C1) / (mu1_sq + mu2_sq + C1)) * cs_map
    return ssim_map


def ssim(X, Y, data_range=255, size_average=True, win_size=11, win_sigma=1.5, win=None,
         K=(0.01, 0.03), nonnegative_ssim=False):
    """NOTE the unusual convention inherited from their code: `size_average=True` returns
    a per-IMAGE value (N,), `size_average=False` returns the full (N,C,h,w) map. Their
    SSIMLoss calls it with size_average=False and then reduces itself."""
    if not X.shape == Y.shape:
        raise ValueError(f"shape mismatch: {X.shape} vs {Y.shape}")
    for d in range(len(X.shape) - 1, 1, -1):
        X, Y = X.squeeze(dim=d), Y.squeeze(dim=d)
    if len(X.shape) not in (4, 5):
        raise ValueError(f"expected 4-d or 5-d, got {X.shape}")
    if win is not None:
        win_size = win.shape[-1]
    if not (win_size % 2 == 1):
        raise ValueError("Window size should be odd.")
    if win is None:
        win = _fspecial_gauss_1d(win_size, win_sigma)
        win = win.repeat([X.shape[1]] + [1] * (len(X.shape) - 1))
    ssim_map = _ssim(X, Y, data_range=data_range, win=win, size_average=False, K=K)
    return torch.mean(ssim_map, dim=[1, 2, 3]) if size_average else ssim_map


# ---- losses, ported verbatim from MedIAnomaly/reconstruction/utils/losses.py ----------
class AELoss(nn.Module):
    """Plain squared error — the 'AE' row of Table 6."""
    def __init__(self, grad_score=False):
        super().__init__()
        self.grad_score = grad_score

    def forward(self, net_in, net_out, anomaly_score=False, keepdim=False):
        x_hat = net_out['x_hat']
        loss = (net_in - x_hat) ** 2
        if anomaly_score:
            if self.grad_score:
                grad = torch.abs(torch.autograd.grad(loss.mean(), net_in)[0])
                return torch.mean(grad, dim=[1], keepdim=True) if keepdim else torch.mean(grad, dim=[1, 2, 3])
            return torch.mean(loss, dim=[1], keepdim=True) if keepdim else torch.mean(loss, dim=[1, 2, 3])
        return loss.mean()


class SSIMLoss(nn.Module):
    """1 - SSIM. The Gaussian window shrinks the map by (win_size-1), hence the
    interpolate back to input size when a pixel map is requested."""
    def __init__(self, win_size=11):
        super().__init__()
        self.win_size = win_size

    def forward(self, net_in, net_out, anomaly_score=False, keepdim=False):
        x_hat = net_out['x_hat']
        net_in_01 = ((net_in + 1) / 2.0).clamp(0., 1.)
        x_hat_01  = ((x_hat + 1) / 2.0).clamp(0., 1.)
        loss = 1. - ssim(net_in_01, x_hat_01, data_range=1., size_average=False, win_size=self.win_size)
        if anomaly_score:
            return torch.mean(F.interpolate(loss, size=net_in.shape[-2:], mode='bilinear'),
                              dim=[1], keepdim=True) if keepdim else torch.mean(loss, dim=[1, 2, 3])
        return loss.mean()


class L1Loss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, net_in, net_out, anomaly_score=False, keepdim=False):
        loss = torch.abs(net_in - net_out['x_hat'])
        if anomaly_score:
            return torch.mean(loss, dim=[1], keepdim=True) if keepdim else torch.mean(loss, dim=[1, 2, 3])
        return loss.mean()


class AEULoss(nn.Module):
    """AE-U: the net also predicts log_var, so the reconstruction error is divided by a
    learned per-pixel variance. That is why it wins on CXR — it learns to stop paying for
    the high-variance regions (ribs, edges) that a plain AE never reconstructs either.
    Returns a 3-tuple in TRAIN mode (loss, recon, log_var) — the driver unpacks it."""
    def __init__(self):
        super().__init__()

    def forward(self, net_in, net_out, anomaly_score=False, keepdim=False):
        x_hat, log_var = net_out['x_hat'], net_out['log_var']
        recon_loss = (net_in - x_hat) ** 2
        loss1 = torch.exp(-log_var) * recon_loss
        loss = loss1 + log_var
        if anomaly_score:
            # scored on loss1 only: the +log_var term is a normalising constant per pixel,
            # not evidence of anomaly
            return torch.mean(loss1, dim=[1], keepdim=True) if keepdim else torch.mean(loss1, dim=[1, 2, 3])
        return loss.mean(), recon_loss.mean().item(), log_var.mean().item()


class VAELoss(nn.Module):
    def __init__(self, kl_weight=0.005, grad=None):
        super().__init__()
        self.kl_weight = kl_weight
        self.grad = grad

    def forward(self, net_in, net_out, anomaly_score=False, keepdim=False):
        x_hat, mu, log_var = net_out['x_hat'], net_out['mu'], net_out['log_var']
        recon_loss = (net_in - x_hat) ** 2
        kl_loss = torch.mean(-0.5 * (1 + log_var - mu ** 2 - log_var.exp()), dim=1)
        loss = recon_loss.mean() + self.kl_weight * kl_loss.mean()
        if anomaly_score:
            if self.grad in ('elbo', 'rec', 'kl', 'combi'):
                target = {'elbo': loss, 'rec': recon_loss.mean(), 'kl': kl_loss.mean(),
                          'combi': kl_loss.mean()}[self.grad]
                g = torch.abs(torch.autograd.grad(target, net_in)[0])
                out = recon_loss * g if self.grad == 'combi' else g
                return torch.mean(out, dim=[1], keepdim=True) if keepdim else torch.mean(out, dim=[1, 2, 3])
            return torch.mean(recon_loss, dim=[1], keepdim=True) if keepdim \
                else torch.mean(recon_loss, dim=[1, 2, 3])
        return loss, recon_loss.mean().item(), kl_loss.mean().item()

### Cell 3.0b — Perceptual loss (AE-PL), the strongest baseline in Table 6
`RelativePerceptualL1Loss` compares VGG19 `relu4_2` features instead of pixels, with
each channel divided by its ImageNet-wide mean/std and the error taken relative to the
feature magnitude.

**Two deliberate deviations from their file**, both forced by "must run as one Kaggle
notebook" and neither of which changes the objective:

1. They copy VGG19's weights into 16 hand-declared conv layers and tap `r42` by name.
   We tap `torchvision.vgg19().features[:23]`, which *is* relu4_2 — same weights, same
   padding (=1), same output. Verified in the assert at the bottom of the cell.
2. Their per-channel normalisation lives in a 106 KB `.pt` file we can't ship. Only the
   512 `r42` means and vars are ever read (default `feature_weights={"r42": 1}`), so
   those 1024 floats are embedded below as a compressed base64 blob. Byte-identical to
   the file's values.

If VGG19 weights can't be downloaded (Kaggle internet off), `HAS_VGG` goes False and
the AE-PL rows are skipped with a message rather than silently substituting something
else.

In [ ]:

import base64, zlib

# r42 (relu4_2) channel means and variances from MedIAnomaly's
# utils/data/vgg19_ILSVRC2012_object_detection_mean_var.pt — first 512 floats are the
# means, next 512 the vars.
_R42_STATS_B64 = (
    "eNoNlXcgFoobhdta97ZLaQhRia6Vle+cbMnK3jOjiCg7fD7jQ6RhFxFRol2o0B7a/UhulNvet27KvQ0/f73/v+c5z1ExcoGn"
    "3lR8TsxFtXMSHkcV4qliJQ4ucIVUUiMOjdDFWlkhFJvaccApAxf3N+H2z0NIXp2PzvvRCPzujzT1cowcWQqpkBLoG3vhg+RN"
    "GL3Ixo7yLKybXYKh3qn4J8wTkUEnUdS2FbnGBWi03IshOtcgqa4DlfnJkHEPwuWLy5GUK8Yy13pc0DfAAeVsfJ56Cis787Hv"
    "7wRkWNZj8f+eCxq3VGBV71N4cihf7gzB18fdeC0swCubo/DXssMl1eNwzRAjQN0buvuOw2zTAnQ1ZWFgWxli3vpgj1QMbt2v"
    "QrGjL9bln8eSOesg01iCpI7tmN6xGYGnD2K57Vo4SR/EZ4O5qK2Pwz6NCGxDGkSPs/EwOwUhd/Ow8S8vfIluQeTCD4IhpTk4"
    "PGIT+rbKIly/XvDDJA3i1CrI0hlG86bg5sdjiM4ohZyiGfZKDYH3oRQ4K/pBaf42BDQ04tmVhSg8Gw67PSWo0qyH+gRNjFur"
    "i+AnL5Bx2wLhb+OhcTQD4bHNWBsUDYFhBladioN7eRrOXXeE+LMs+qRO4uQOExRtLMGeoVV4IXsG/RGpeNLxAGKPvej7zRft"
    "MqnQiQ2F8u6zaMh+g/uBYoT/W4N9D/JR67YbfQMpGHO0AkalRXD/txXwuy74UauP0kuuMHEfj0YPMc741mGKehOCGrZB0z0Q"
    "Sa3bEd9bi5AHQtjFHMUGOWeY3bqE2pjreDFJiIlflSFsDkfslUyUNojg7GiD/atTIZGZji978jG0uQCXpdbgt51OUPSyxIQR"
    "J+HU5INc3x3w105BRVsdnjU1wndkFmY4aMH0SQ6mrz6Oa2mVKFdww8shgAE0MWxkDOTjt+Pd6HuCb6eLkOhfiTavMkjWO0Ar"
    "rBJXby1H+fWleHLGFj8ObMARu5M4d7UcQWVpMDGTwdIyIc5Gu8NAKx1+3XG4eWAvJsTl4KjCPnidaMT5NiNkdLoh2LYIPzwS"
    "8Hz7XPQoz8ae8rcCo7xSvNcrhujwK0jbq6PfIwK3D1Uj+UohenebY0F2MCSivTBhuTS6+u8JJNeH43pXPV6p1cDp3CbkZHQJ"
    "ZlXqorR1PSa3WGKSWgnGVVzACG9XZDadRWldAZ78ewIt4n7UdGcg0c0JLaoPIfVPJoLjzTHdPgOnt+bAovgr7hRWI+JBCrxK"
    "TmPosgo43zDBuYDzWGTrgqd2uSjpiIXpVeJXawDem1dAKeshFjUp4HqCGHEyUZgnKcIxX3ucai7HurATkDl+FHLtCriUpInK"
    "3KvI0S5CdHQCSgZeCj6cuiNYIQpBiJYy3irG4InnLhSM1cW4PbXIcT6LueMj8CinBGPf5GP3pxZUZ6UhXGE7nrR9F/zoPgV7"
    "T2/k/BJjQFIJMns24tGOVth80YHkqEiohPkhLLIVS0zrsOzfUfiUmo+xQ28KtMNyEN6ehDt7bPEoLnXQE4G4bJyElItG+HX3"
    "Nkw0C9BzKwG7Y3Jx+40nOqe0Iup8PlR10yAcL4dl9mNQkKgB5cImVGoEI3/uFVwYfR+qLTk4e2opJH8XY5T/RBxqEsP6eK8g"
    "c1gyHI4JBznMgsOdGty2yoF2Yw1snAcZca+Fb0cplE4Ssx12wDZgJ56aNMDS1hZn8//CnGX3ceuxLroTffAmqw69oy9g9LBU"
    "FK/cCdMHd9EcYwXfC4sRvt4Dt9o3Y4Z6MrZWtsBH/6FgfVYYGi6lw3qYJkx2nIbrpkDEjUlBMfzxonYEXA4eQ9GGQR8Ub8Ih"
    "5ZOQ9t2OaoWdkFG6gXd+SdjokIS+wHNo+xkMb4EQLa1n4P8iBldki9HudRbdc5JR84cH/FmD6/JWg/nvQ9opD1iP8UBViTqe"
    "aejAV2I39D7shrFDJd6cm4ermy8LhNZrkHg7EX+Z5+P52CpBhHoa5kRFQNGhAhoDvyNwdApUOg0xcko2atwMIL3sAnaIr0NF"
    "0hYigyrMOueI/Kx8FEQ6INYnFad3q2BMoQgtS9bgW1k6OmRWoqd3C56Hn0L6wxJkm21C+qlr+D2jDooduzDm/ST4dBVDLzkL"
    "HQIRlI4XYXlKKa47zETmCkfs7EvCRcsWOK88Df/+rcgcfgMZrhdwrrsQn+rkMXPKkUHX5sPspDkMAjMh23ccC/++DO95g5w8"
    "rxG037GHRMN5fHuwBafuVCN2rwiG6mbwH7kd4q5prPlzK6zcBv3imAfh6Vasnf0CetXAl33vBQXPdiEoqBgK37JQ5LMI8pMW"
    "4li7EDdPHMAtxVL0J7gira0Q3l7Z+KNqOkZtzUSbVD0+eBnD4LQ7hLVjGF13FBrSHuhjNxaa1KJwvyx+Zk/GtJ5ItE2OhF1A"
    "EcYnnMQqfV0cspGB8x05dKolwz+0EWFlAXD+JxjxSzxQfskQljONMCrumuCo3jz0zFbAV0EWKuTLUbkzDvsH2Woqs+BBR1Oq"
    "zfRjlrYcI5MzmPwglDWjZGg2PIVxVKdotoh/rnalt4UlX+/N4BtnH9bU+jAy155PrQy5+qktHdLNmN/uR4mXmjQ5kc2gE2kU"
    "di5jYrsmN+7yZryzFg8ftqOMQi5j1oRSLS6O/73YwJwrs+j0QJMyn+15QleLLV/sOGJgBcVLtdnpYsWrOcl0mWo+eA05fHIa"
    "JZJ7oXMsk9MupLDHpZyT7dyZtyKeGpIe3Pg6njVPVKkx3IabN5qw96cxi+X8+d1Jnh+7RfSosOXO1y6Mt9Xj3IlCXjgfzZmj"
    "wlkXqsG4+iKaJIo59EkQzVurqBibzI91Xtz8YTjzDbzY7+lGHbMV7Jm0jiLDVXT030jvWUqM74qg/dIXgx4VMEEqgqvWjGa/"
    "7VhqLVXnPRt3xhcu5bwaVTbfiGZPkysXJpjyyBBpVnr48IiyN4/2OfLb32Ke71HkkW8CWnvZ0e6tI699Wswbn+Yw8WM2JX7Y"
    "c6LIjjOLI1jquJX/uPpxWLMGo2fGM9xYne+tFzHNdCrTpZxYJr+OBluEvJeQxEJ1MUvstlDvrwSGPI+jo4kpw/Zq0UPejms/"
    "OjEgRsQLz2z4PS2JK/TNaOMRx13fQ7m8Scw3ORt4PrSJ6cUzqBghw1tRmhx9cQDnR9hR/nMyD70SsfK6Fc/dU6OJvieTPTOY"
    "pGZGr6id/Nrrxv9dTWDyURH9PjhxlrkPq6+ZcuC7A+XtFvJxmzaFzd6sm6pEu0NraV7gQKMpTpywS4e1P1UZFp/H9J+WHKVi"
    "wm7rlexQ8OYTNzEdgu14e6IiQ5/68HJkEpXoSd/5K9kW7cotu+RpOmMFzZ97803ef4j7N55y3UK+N4vjU4tQLii35/KHmrS+"
    "okr/Nl22Gpmw0imTGbcjuV5Wl0/zZ/KxyIaT/lKivosRVwp2UskqmM+kbCjdUkXTl0ns5Qw+OyfFg23JnFQWRrcfUhzwXcLv"
    "p7sw/pkPm2Us+PN1KLfJzqClii5HrM5n6llHptjP4XVZHVZVyLF4twRr/n2DyGWGFP70ZsrMWFYZWFPJ7yt2NOszbZo7uV+N"
    "7yQM2X3Hj9YD4CiDJD7a4kKZy/7cl7mZM+w9qJBqxqyzm+lwxpajDpmyt8GAhyxSuD21lNX33OhgpU/94E28eiyJYqNFXL4j"
    "m64hc7gzJpGLNMx4p9mQvhNm8nb5Sh4xF/HjwGTmL1hFly/m/KXnyD+eKHHl6AC+mhfMYX5+7Hg2jnlq8tR8kEdVLTeeFrpx"
    "W8pzSD5+gTQ3AS07lWgp0uGoSjvarJrGX1EZ7JBNpJK/MZv3iaioL2S/Uwg3HHag9Me1TO8cwBALf26RleaH/Qb8ff0EThrt"
    "yuAQEZ0yB/+UR9ZbrWSDnifvD0ugpdE9eBaYULrzKjyuaXFzrBULthhyQ542U5VNuO1qEvtGL+RizWTuGunD0GxVrjxowWHD"
    "klhpnM4z6hEssg7j1keKzLygRBc/bXqcCKHamtWs27uF3+8ms1TbixP1ZnGWngnHX5xOAzMXbrZ4h1UlhqzrsuQZLWPK5Ym4"
    "Sd2DPVmBzOY6LtCz4R0jfQb/PY3nJgz2TMmK52/E8c8+bToXbeZsryQOlBnyztB1DPvsxA99Qs63XsWE3WbMXFbClpm6fPSH"
    "Il9+s6BOqAEvizV5QyqT41XHUOKqIoWSqRxw1GbIyc0s69Gj2Mma++qXcNvskbRa5zuYtwMDfwnY0CukpI0tcwJSWaksZLiM"
    "Cps+2XPu/iiGdLkxxtiQb8YJOaLYl3cfi6jyXzKfNXlyQa8V/Vs30jBRj90lUVwRt5BYoUJbVwnGChS4aagRx16z5IbHGxiV"
    "s4jiu114RnMePKrKi/FOrFo1gw0TrDipfzl/7s7n8F+zOK7Sn1nzdWmzbS2dg8xobhhAcb6Q6lpazFLzZf9EV86dH8jHjUr8"
    "5WvHnAOrB90UxOmvjXiz35JPfTWp+JsmOTaRs8q86GunRyXnQornJdPzH2uuyZXnCQtnJnQlsLpTn40TbFlW5sO3UbMp98iR"
    "s6vB1odJHKmxngqvQ3hfL5Jylbks9FHhhgFZuv/nzo1R0TzVo0knGxc+ig1im+5GrjAJ5L1f/8FLSo7xlkEMSbWhslMiq6vt"
    "WFGUxPYpkYy/e4CPBreyvdKacnMGO6jtTrWGTTTwXMDV379D891a7p+3ho4jPLm8fgr9Ximw9aIuD98MY2SYFb111bjuYjCv"
    "VQXxtxBpbh9kPK/RgpPkZbjCVZlhX6u5TSKcic+06NYnplyAJ8sz5tAr4HeuKycnZQhoMiyMsxbHMiRrPrumSlOOixkyIOS7"
    "7VkcnF7Od1Zm2zJ3Pl40jzrOxlxbMZS+h8fxa+44HpRYyIyIVQzs+INz9B35fwmht4A="
)


def _r42_mean_var():
    a = np.frombuffer(zlib.decompress(base64.b64decode(_R42_STATS_B64)), dtype=np.float32)
    assert a.size == 1024, a.size
    m = torch.from_numpy(a[:512].copy()).reshape(1, 512, 1, 1)
    v = torch.from_numpy(a[512:].copy()).reshape(1, 512, 1, 1)
    return m, v


class RelativePerceptualL1Loss(nn.Module):
    """MedIAnomaly's AE-PL objective: relative L1 on channel-normalised VGG19 relu4_2."""
    IMAGENET_MEAN = (0.485, 0.456, 0.406)
    IMAGENET_STD  = (0.229, 0.224, 0.225)

    def __init__(self):
        super().__init__()
        vgg = tv_models.vgg19(weights=tv_models.VGG19_Weights.IMAGENET1K_V1)
        self.features = vgg.features[:23].eval().to(device)   # up to and incl. relu4_2
        for p in self.features.parameters():
            p.requires_grad = False
        m, v = _r42_mean_var()
        self.register_buffer('feat_mean', m)
        self.register_buffer('feat_var',  v)
        self.register_buffer('vgg_mean', torch.tensor(self.IMAGENET_MEAN).reshape(1, 3, 1, 1))
        self.register_buffer('vgg_std',  torch.tensor(self.IMAGENET_STD).reshape(1, 3, 1, 1))
        self.to(device)

    def _preprocess(self, x):
        if x.shape[1] != 3:
            x = x.expand(-1, 3, -1, -1)
        x = x * 0.5 + 0.5                      # [-1,1] -> [0,1]
        return (x - self.vgg_mean) / self.vgg_std

    def _relative_l1(self, fx, fy):
        # relative to the magnitude of the REAL image's features, detached so the
        # denominator is a scale factor and not a second gradient path
        means = torch.abs(fx).mean(3).mean(2).mean(1).detach()
        return torch.abs(fx - fy) / means.reshape(-1, 1, 1, 1)

    def forward(self, net_in, net_out, anomaly_score=False, keepdim=False):
        y = net_out['x_hat']
        fx = self.features(self._preprocess(net_in))
        fy = self.features(self._preprocess(y))
        fx = (fx - self.feat_mean) / self.feat_var
        fy = (fy - self.feat_mean) / self.feat_var
        loss = self._relative_l1(fx, fy)
        if anomaly_score:
            if keepdim:
                loss = F.interpolate(loss, size=net_in.shape[-2:], mode='bilinear')
                return torch.mean(loss, dim=[1], keepdim=True)
            return torch.mean(loss, dim=[1, 2, 3])
        return loss.mean()


HAS_VGG = True
try:
    _pl_probe = RelativePerceptualL1Loss()
    with torch.no_grad():
        _z = torch.zeros(2, 1, IMAGE_SIZE, IMAGE_SIZE, device=device)
        _f = _pl_probe.features(_pl_probe._preprocess(_z))
    assert _f.shape[1] == 512, _f.shape
    print(f'perceptual loss ready — relu4_2 feature map {tuple(_f.shape)} '
          f'(mean stats: {float(_pl_probe.feat_mean.mean()):.4f})')
    del _pl_probe, _z, _f
except Exception as e:
    HAS_VGG = False
    print(f'VGG19 unavailable ({type(e).__name__}: {e}) — AE-PL rows will be SKIPPED.\n'
          '  On Kaggle: enable "Internet" in the notebook settings, or attach a\n'
          '  torchvision-weights dataset.')

---
## **Cell 3.1** — Method registry
One table mapping a method name to (backbone, training loss, scoring loss, extras).
Everything downstream reads this table, so adding a method is a one-line change and
no experiment cell ever constructs a network by hand.

**The AEU/VAE depth trap.** `AE.__init__` defaults `en_num_layers=1`, but `AEU` and
`VAE` default both depths to `None` and pass them straight through to `BasicBlock`,
which then does `range(None)` and dies with
`TypeError: 'NoneType' object cannot be interpreted as an integer` — a message that
points nowhere near the real cause. Their own `base_worker.set_network_loss` always
passes the depths explicitly, so the defaults are simply dead. `build_net` below does
the same and asserts, which is the only place in this notebook a network is built.

In [ ]:

# ---- VAE, ported verbatim from MedIAnomaly/reconstruction/networks/vae.py -------------
class VAE(AE):
    def __init__(self, input_size=64, in_planes=1, base_width=16, expansion=1, mid_num=2048,
                 latent_size=16, en_num_layers=None, de_num_layers=None):
        super(VAE, self).__init__(input_size, in_planes, base_width, expansion, mid_num,
                                  latent_size, en_num_layers, de_num_layers)
        self.bottle_neck = VaeBottleNeck(4 * base_width * expansion, feature_size=self.fm,
                                         mid_num=mid_num, latent_size=latent_size)

    def forward(self, x):
        en1 = self.en_block1(x)
        en2 = self.en_block2(en1)
        en3 = self.en_block3(en2)
        en4 = self.en_block4(en3)
        bottle_out = self.bottle_neck(en4)
        de4, mu, log_var = bottle_out['out'], bottle_out['mu'], bottle_out['log_var']
        de3 = self.de_block1(de4)
        de2 = self.de_block2(de3)
        de1 = self.de_block3(de2)
        x_hat = self.de_block4(de1)
        return {'x_hat': x_hat, 'log_var': log_var, 'mu': mu,
                'en_features': [en1, en2, en3], 'de_features': [de1, de2, de3]}


_NET_CLASSES = {'ae': AE, 'aeu': AEU, 'vae': VAE}


def build_net(kind):
    """The ONLY place a backbone is instantiated. Depths are always passed explicitly —
    see the AEU/VAE None-default trap in the markdown above."""
    if kind == 'unet':
        # DAE does NOT use the AE backbone. base_worker.set_network_loss builds
        # UNet(in_channels, n_classes) for 'dae', i.e. the class defaults depth=5, wf=6.
        # The paper attributes DAE's segmentation win largely to this "customised UNet",
        # so substituting an AE here would not be a DAE.
        #
        # HEADS-UP for the write-up: Table 6 lists DAE as 2.79M params / 2.15 GFLOPs, but
        # those defaults give 31.0M params. The FLOPs side does check out (a depth-5 wf-6
        # UNet at 64px is ~1.07 G MACs ~= 2.15 GFLOPs, and thop reports MACs), so the
        # architecture is right and it is the params column that cannot be reconciled —
        # no (depth, wf) combination yields 2.79M. We follow the code, not the table, and
        # report our measured 31.0M. Worth stating explicitly rather than quietly
        # matching a number we cannot reproduce.
        return UNet(in_channels=1, n_classes=1,
                    depth=DAE_UNET_DEPTH, wf=DAE_UNET_WF).to(device)
    assert EN_DEPTH is not None and DE_DEPTH is not None, \
        'EN_DEPTH/DE_DEPTH must be ints — AEU and VAE forward None straight into range()'
    cls = _NET_CLASSES[kind]
    return cls(input_size=IMAGE_SIZE, in_planes=1, base_width=BASE_WIDTH, expansion=1,
               mid_num=HIDDEN_NUM, latent_size=LATENT_DIM,
               en_num_layers=EN_DEPTH, de_num_layers=DE_DEPTH).to(device)


_LOSS_CACHE = {}


def build_loss(name):
    """name -> criterion instance. Kept separate from build_net so experiment A1 can
    pair any training loss with any scoring loss.

    The perceptual criterion is cached: it wraps a 20M-parameter frozen VGG19, and A1
    asks for it once per grid cell. Rebuilding it each time re-allocated the whole
    feature extractor on the GPU — pure waste, and a plausible OOM on a 16 GB Kaggle
    accelerator. It holds no per-run state (frozen weights and two constant buffers),
    so a single shared instance is safe."""
    if name == 'perceptual':
        if 'perceptual' not in _LOSS_CACHE:
            _LOSS_CACHE['perceptual'] = RelativePerceptualL1Loss()
        return _LOSS_CACHE['perceptual']
    if name == 'l2':             return AELoss()
    if name == 'l2grad':         return AELoss(grad_score=True)
    if name == 'l1':             return L1Loss()
    if name == 'ssim':           return SSIMLoss()
    if name == 'aeu':            return AEULoss()
    if name == 'vae':            return VAELoss()
    if name == 'vaegrad-rec':    return VAELoss(grad='rec')
    if name == 'vaegrad-combi':  return VAELoss(grad='combi')
    raise KeyError(f'unknown loss {name!r}')


# Losses whose TRAIN-mode call returns (loss, extra1, extra2) instead of a bare scalar.
_TUPLE_LOSSES = {'aeu', 'vae'}
# Losses whose SCORE needs a gradient w.r.t. the input (so no torch.no_grad at eval).
_GRAD_LOSSES  = {'l2grad', 'vaegrad-rec', 'vaegrad-combi'}

# method name -> everything needed to run it. `train_loss`/`score_loss` are separate
# fields precisely so A1 can decouple them; for every baseline they are equal, which is
# the standard practice this benchmark (and the whole literature) assumes.
METHODS = {
    #                    net     train_loss    score_loss    extras
    'ae':         dict(net='ae',  train_loss='l2',         score_loss='l2'),
    'ae-l1':      dict(net='ae',  train_loss='l1',         score_loss='l1'),
    'ae-ssim':    dict(net='ae',  train_loss='ssim',       score_loss='ssim'),
    'ae-pl':      dict(net='ae',  train_loss='perceptual', score_loss='perceptual'),
    'aeu':        dict(net='aeu', train_loss='aeu',        score_loss='aeu'),
    'vae':        dict(net='vae', train_loss='vae',        score_loss='vae'),
    'dae':        dict(net='unet', train_loss='l2',        score_loss='l2',
                       denoise=True, noise_res=16, noise_std=0.2),
}

# MedIAnomaly Table 6, RSNA column (AUC ± sd, AP ± sd over 3 seeds). Our target.
TABLE6_RSNA = {
    'ae':      (67.5, 0.9, 66.7, 0.5),
    'ae-l1':   (68.1, 0.4, 67.9, 0.4),
    'ae-ssim': (80.9, 0.3, 78.6, 0.3),
    'ae-pl':   (87.5, 0.2, 84.8, 0.6),
    'vae':     (67.9, 0.8, 67.0, 0.8),
    'aeu':     (86.5, 0.9, 84.4, 1.1),
    'dae':     (86.1, 0.7, 83.3, 1.0),
    # score-swap rows: same backbone AND same training loss as ae / vae (Table 5)
    'ae-grad':        (73.7, 1.0, 71.0, 1.1),
    'vae-grad-rec':   (71.6, 0.8, 69.3, 0.9),
    'vae-grad-combi': (67.4, 0.3, 67.3, 0.4),
}

print(f'{len(METHODS)} methods registered: {", ".join(METHODS)}')

---
## **Cell 3.2** — Training / evaluation driver
A single function that runs any registry entry end to end and stores the result via
`save_run`. Behaviour it enforces, each for a reason we hit the hard way earlier:

* **`net.eval()` before scoring, always.** Scoring in train mode makes BatchNorm use
  batch statistics *and* mutates its running statistics, which silently corrupts the
  weights being saved. That single bug produced a 0.85-vs-0.95 discrepancy once.
* **Per-run seeding of weights, batch order and DAE noise** from `seed`, so a "seed
  spread" is a real measure of run-to-run variance and not of leftover global state.
* **Skip / restore before training**: an existing local run is reused, otherwise wandb
  is tried, otherwise it trains. `load_run` refuses a record made under a different
  `config_fingerprint`, so a reused run can never be from different hyperparameters.
* **Scores are always computed by `score_criterion`**, which may differ from the
  training criterion — that is the whole mechanism of experiment A1.

In [ ]:

from sklearn.metrics import roc_auc_score, average_precision_score


def add_noise(x, noise_res, noise_std):
    """DAE corruption, ported from MedIAnomaly dae_worker.add_noise (Kascenas et al.).
    Coarse noise (16x16) upsampled to full res and randomly rolled — NOT per-pixel
    Gaussian. That is the point: coarse noise forces the net to use context to repair a
    blob, which is what makes it transfer to blob-shaped pathology."""
    ns = torch.normal(mean=torch.zeros(x.shape[0], x.shape[1], noise_res, noise_res),
                      std=noise_std).to(x.device)
    ns = F.interpolate(ns, size=x.shape[-1], mode='bilinear', align_corners=True)
    roll_x = random.choice(range(x.shape[-2]))
    roll_y = random.choice(range(x.shape[-1]))
    ns = torch.roll(ns, shifts=[roll_x, roll_y], dims=[-2, -1])
    ns = (ns - 0.5) * 2          # their `config.center` branch, always taken for CXR
    return x + ns, ns


@torch.no_grad()
def _forward_scores(net, criterion, x, batch_size=256):
    """Per-image anomaly scores for a whole tensor. net MUST already be in eval mode."""
    out = []
    for i in range(0, len(x), batch_size):
        xb = x[i:i + batch_size].to(device)
        out.append(criterion(xb, net(xb), anomaly_score=True).cpu())
    return torch.cat(out).numpy()


def _forward_scores_grad(net, criterion, x, batch_size=64):
    """Same, for gradient-based scores, which need graph + input grads."""
    out = []
    for i in range(0, len(x), batch_size):
        xb = x[i:i + batch_size].to(device).requires_grad_(True)
        out.append(criterion(xb, net(xb), anomaly_score=True).detach().cpu())
    return torch.cat(out).numpy()


def evaluate_scores(scores, y):
    """Image-level metrics, matching AEWorker.evaluate."""
    return {
        'AUC': float(roc_auc_score(y, scores)),
        'AP':  float(average_precision_score(y, scores)),
        'normal_score':   float(np.mean(scores[y == 0])),
        'abnormal_score': float(np.mean(scores[y == 1])),
    }


def train_and_eval(method, seed=TRAIN_SEED, *, extra_params=None, epochs=None,
                   train_loss=None, score_loss=None, save_weights=True, verbose=True):
    """Run one (method, params, seed) and persist it. Returns the manifest dict."""
    spec = dict(METHODS[method])
    if train_loss is not None: spec['train_loss'] = train_loss
    if score_loss is not None: spec['score_loss'] = score_loss
    params = dict(extra_params or {})
    # A shortened run must never share an id with the full-length one, or a 2-epoch smoke
    # run would later be reused as if it were the real 250-epoch result. config_fingerprint
    # records the GLOBAL epoch count and cannot see this per-call override, so encode it here.
    if epochs is not None and epochs != EPOCHS:
        params['ep'] = epochs

    # --- skip / restore -------------------------------------------------------------
    rid = run_id(method, seed, **params)
    if run_exists(rid) or (USE_WANDB and fetch_run(rid)):
        try:
            man, _ = load_run(rid)
            if verbose:
                print(f"{rid}: reused  AUC={man['metrics']['AUC']:.4f}")
            return man
        except (RuntimeError, FileNotFoundError) as e:
            print(f'{rid}: stored record unusable, retraining\n    {e}')

    if spec['train_loss'] == 'perceptual' or spec['score_loss'] == 'perceptual':
        if not HAS_VGG:
            print(f'{rid}: SKIPPED — perceptual loss needs VGG19 weights (see Cell 3.0b)')
            return None

    # --- deterministic setup --------------------------------------------------------
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    net = build_net(spec['net'])
    train_crit = build_loss(spec['train_loss'])
    score_crit = train_crit if spec['score_loss'] == spec['train_loss'] else build_loss(spec['score_loss'])
    opt = Adam(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    g = torch.Generator().manual_seed(seed)
    loader = DataLoader(TensorDataset(X_TRAIN), batch_size=BATCH_SIZE, shuffle=True,
                        drop_last=False, generator=g)

    n_epochs = epochs if epochs is not None else EPOCHS
    tuple_loss = spec['train_loss'] in _TUPLE_LOSSES
    n_params = sum(p.numel() for p in net.parameters())
    if verbose:
        print(f"{rid}: {spec['net'].upper()} {n_params:,} params | train={spec['train_loss']} "
              f"score={spec['score_loss']} | {n_epochs} epochs")

    # --- train ----------------------------------------------------------------------
    epoch_loss, t0 = [], time.time()
    for ep in range(1, n_epochs + 1):
        net.train()
        tot, n = 0.0, 0
        for (xb,) in loader:
            xb = xb.to(device, non_blocking=True)
            net_in = xb
            if spec.get('denoise'):
                noisy, _ = add_noise(xb, spec['noise_res'], spec['noise_std'])
                out = net(noisy)          # corrupted IN ...
            else:
                out = net(xb)
            loss = train_crit(net_in, out)    # ... clean as TARGET
            if tuple_loss:
                loss = loss[0]
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * xb.size(0); n += xb.size(0)
        epoch_loss.append(tot / n)
        if verbose and (ep == 1 or ep % max(1, n_epochs // 10) == 0 or ep == n_epochs):
            print(f'    ep {ep:>4}/{n_epochs}  loss {epoch_loss[-1]:.5f}  '
                  f'({time.time() - t0:.0f}s)')

    # --- evaluate -------------------------------------------------------------------
    net.eval()                       # never score in train mode (see markdown)
    if spec['score_loss'] in _GRAD_LOSSES:
        scores = _forward_scores_grad(net, score_crit, X_TEST)
    else:
        scores = _forward_scores(net, score_crit, X_TEST)
    metrics = evaluate_scores(scores, Y_TEST)
    metrics['train_loss_final'] = epoch_loss[-1]
    metrics['n_params'] = n_params
    metrics['minutes'] = (time.time() - t0) / 60
    if verbose:
        print(f"    -> AUC {metrics['AUC']:.4f}  AP {metrics['AP']:.4f}  "
              f"({metrics['minutes']:.1f} min)")

    return save_run(rid, method=method, seed=seed,
                    params={**params, 'train_loss': spec['train_loss'],
                            'score_loss': spec['score_loss']},
                    metrics=metrics, epoch_loss=epoch_loss,
                    arrays={'scores': scores, 'labels': Y_TEST},
                    weights={'net': net.state_dict()} if save_weights else None,
                    extra={'spec': {k: v for k, v in spec.items()}})


def rescore(base_method, score_loss, seed=TRAIN_SEED, *, name=None, verbose=True):
    """Score an ALREADY-TRAINED model with a different anomaly-score function. No
    training happens — the base run's stored weights are reused.

    This is not a shortcut, it is what the score-swap methods actually are: Table 5 gives
    AE-Grad the AE backbone and the AE objective, changing only the score, and likewise
    for both VAE-Grad variants. Training them from scratch would differ from `ae`/`vae`
    only by the random seed. Every off-diagonal cell of A1 is the same operation.

    Note on batching: a gradient score takes autograd.grad of `loss.mean()`, so scoring B
    images at once divides every score by exactly B relative to their batch_size=1 test
    loader. AUC and AP are rank statistics, so this is invisible to both — verified —
    but do not compare raw gradient-score magnitudes across batch sizes."""
    name = name or f'{base_method}-{score_loss}'
    rid  = run_id(name, seed)
    if run_exists(rid):
        try:
            man, _ = load_run(rid)
            if verbose:
                print(f"{rid}: reused  AUC={man['metrics']['AUC']:.4f}")
            return man
        except (RuntimeError, FileNotFoundError) as e:
            print(f'{rid}: stored record unusable, recomputing\n    {e}')

    if score_loss == 'perceptual' and not HAS_VGG:
        print(f'{rid}: SKIPPED — perceptual score needs VGG19 (see Cell 3.0b)')
        return None

    base = train_and_eval(base_method, seed=seed, verbose=verbose)   # trains only if needed
    if base is None:
        return None

    net = build_net(METHODS[base_method]['net'])
    load_run(run_id(base_method, seed), models={'net': net})         # also calls .eval()
    fn = _forward_scores_grad if score_loss in _GRAD_LOSSES else _forward_scores
    scores = fn(net, build_loss(score_loss), X_TEST)
    metrics = evaluate_scores(scores, Y_TEST)
    metrics['n_params'] = base['metrics'].get('n_params', float('nan'))
    del net
    if verbose:
        print(f"{rid}: rescored {base_method} with {score_loss} -> "
              f"AUC {metrics['AUC']:.4f}  AP {metrics['AP']:.4f}")
    # deliberately stores NO weights: this run owns no model, it re-scores another's.
    # `base_run` keeps that dependency explicit and auditable.
    return save_run(rid, method=name, seed=seed,
                    params={'train_loss': METHODS[base_method]['train_loss'],
                            'score_loss': score_loss},
                    metrics=metrics, arrays={'scores': scores},
                    extra={'base_run': run_id(base_method, seed), 'base_method': base_method})



# R — Does the rescore survive a change of resolution?

The paper reports the SSIM rescore at $64\times64$: **+9.08** on RSNA, **-2.37** on
VinDr-CXR, **-10.54** on LAG. Limitation 1 then makes a falsifiable prediction. SSIM's
window is a fixed $11\times11$ Gaussian, so doubling the image halves the window's
reach relative to image content; the $64$px bandwidth equivalent of the $128$px default
is $\sigma \approx 0.75$, measured at **-3.31 / -8.77 / -18.39**. The paper therefore
predicts the RSNA gain *vanishes rather than merely shrinks*.

This notebook tests that directly. It trains ONLY the pixel-objective autoencoder, at
$128\times128$, three seeds, then rescores those same frozen weights with SSIM at the
library default. Nothing else in the grid is rebuilt.

**Run one dataset per session.** Set `DATASET` in Cell 0.0 to `RSNA`, then `VinCXR`,
then `LAG`. Restart-and-run-all each time. Roughly 30 min per dataset on a T4.

In [ ]:
# The single way this experiment can silently produce meaningless numbers: the loader
# resizes whatever it finds to IMAGE_SIZE, so pointing it at a pre-resized 64px copy of
# the data would UPSAMPLE to 128 and measure bilinear interpolation, not resolution.
# MedIAnomaly-Data ships at 512x512 (RSNA, VinDr-CXR) and 500x500 (LAG); a folder named
# MedIAnomaly-Data-64 is a downsampled convenience copy and is NOT usable here.
from PIL import Image as _Im
import glob as _glob

_sample = sorted(_glob.glob(os.path.join(DATA_ROOT, DATASET, 'images', '*')))[:1]
assert _sample, f'no images found under {os.path.join(DATA_ROOT, DATASET, "images")}'
_w, _h = _Im.open(_sample[0]).size
print(f'DATA_ROOT   : {DATA_ROOT}')
print(f'source size : {_w}x{_h}   target IMAGE_SIZE: {IMAGE_SIZE}')
assert min(_w, _h) >= IMAGE_SIZE, (
    f'SOURCE TOO SMALL: {_w}x{_h} < {IMAGE_SIZE}. This would upsample and the result '
    f'would be meaningless. Point MEDIANOMALY_DATA at the full-resolution '
    f'MedIAnomaly-Data (512px), not MedIAnomaly-Data-64.')
assert X_TRAIN.shape[-1] == IMAGE_SIZE, \
    f'X_TRAIN is {X_TRAIN.shape[-1]}px but IMAGE_SIZE is {IMAGE_SIZE} — rerun Cell 2.2'
print(f'OK: downsampling {_w}px -> {IMAGE_SIZE}px, same direction as the 64px runs.')

In [ ]:
SEEDS = [42, 43, 44] if not SAMPLE_MODE else [42]

print(f'{DATASET} at {IMAGE_SIZE}px, seeds {SEEDS}, {EPOCHS} epochs')
print('train_and_eval("ae") is called inside rescore(), so each seed trains once.\n')

for _s in SEEDS:
    rescore('ae', 'ssim', seed=_s)          # trains the base AE if it is not stored yet

In [ ]:
import numpy as _np, json as _json

# Published, for comparison only. 64px headline and the sigma=0.75 prediction that the
# 128px default is scale-equivalent to. Both are in the paper; neither is recomputed here.
PAPER_64PX      = {'RSNA': +9.08, 'VinCXR': -2.37, 'LAG': -10.54}
PAPER_PREDICTED = {'RSNA': -3.31, 'VinCXR': -8.77, 'LAG': -18.39}

_runs = completed_runs()
_get  = lambda m: _np.array([float(_runs.loc[_runs.run_id == run_id(m, s), 'AUC'].iloc[0]) * 100
                             for s in SEEDS
                             if (_runs.run_id == run_id(m, s)).any()])
_l2, _ssim = _get('ae'), _get('ae-ssim')
assert len(_l2) == len(_ssim) == len(SEEDS), \
    f'expected {len(SEEDS)} seeds, got l2={len(_l2)} ssim={len(_ssim)}'

_per_seed = _ssim - _l2
_delta    = float(_per_seed.mean())

print('=' * 70)
print(f'{DATASET}  —  SSIM rescore minus pixel score, at {IMAGE_SIZE}px')
print('=' * 70)
for _s, _a, _b in zip(SEEDS, _l2, _ssim):
    print(f'  seed {_s}:  l2 {_a:6.2f}   ssim {_b:6.2f}   delta {_b - _a:+7.2f}')
print(f'\n  mean delta at {IMAGE_SIZE}px : {_delta:+.2f}   (population sd {_per_seed.std():.2f})')
print(f'  paper, same rescore at 64px: {PAPER_64PX[DATASET]:+.2f}')
print(f'  paper predicted for 128px  : {PAPER_PREDICTED[DATASET]:+.2f}')
_signs_agree = bool((_per_seed > 0).all() or (_per_seed < 0).all())
print(f'  all three seeds agree in sign: {_signs_agree}')
print()
if _delta * PAPER_64PX[DATASET] < 0:
    print('  -> SIGN FLIPPED relative to 64px. The prediction holds for this dataset.')
elif abs(_delta) < abs(PAPER_64PX[DATASET]):
    print('  -> Same sign, smaller magnitude: the effect SHRANK but did not vanish.')
    print('     The paper predicted vanishing; this is a partial miss and must be reported.')
else:
    print('  -> Same sign, not smaller. The prediction FAILS for this dataset.')
    print('     Report it: Limitation 1 must be rewritten, not quietly dropped.')

_out = {'dataset': DATASET, 'image_size': IMAGE_SIZE, 'epochs': EPOCHS,
        'seeds': list(SEEDS), 'auc_l2': _l2.tolist(), 'auc_ssim': _ssim.tolist(),
        'per_seed_delta': _per_seed.tolist(), 'mean_delta': _delta,
        'seeds_agree_in_sign': _signs_agree,
        'paper_64px': PAPER_64PX[DATASET], 'paper_predicted_128px': PAPER_PREDICTED[DATASET],
        'run_version': RUN_VERSION}
with open(f'{OUTPUT_DIR}/res128_{DATASET}.json', 'w') as _f:
    _json.dump(_out, _f, indent=2)
print(f'  written: {OUTPUT_DIR}/res128_{DATASET}.json')

In [ ]:

INCLUDE_WEIGHTS = True        # False -> scores + manifests + tables only (~50x smaller)

import zipfile, datetime, shutil


def _human(nbytes):
    for u in ['B', 'KB', 'MB', 'GB']:
        if nbytes < 1024 or u == 'GB':
            return f'{nbytes:.1f} {u}'
        nbytes /= 1024


def write_summary(path):
    """A plain-text record of what this session produced, so the zip is readable on its
    own without re-running anything."""
    runs = completed_runs()
    L = []
    A = L.append
    A('=' * 78)
    A(f'DL PROJECT RESULTS BUNDLE')
    A(f'dataset      : {DATASET}')
    A(f'run version  : {RUN_VERSION}')
    A(f'generated    : {datetime.datetime.now().isoformat(timespec="seconds")}')
    A(f'seeds        : {SEEDS}')
    A(f'epochs       : {EPOCHS}   sample mode: {SAMPLE_MODE}')
    A(f'runs stored  : {len(runs)}')
    A('=' * 78)
    if not runs.empty:
        A('')
        A('ALL RUNS (sorted by AUROC)')
        A('-' * 78)
        cols = [c for c in ['run_id', 'method', 'seed', 'AUC', 'AP', 'n_params',
                            'minutes'] if c in runs]
        A(runs[cols].sort_values('AUC', ascending=False).to_string(index=False))
        A('')
        A('BY METHOD (mean +- population sd over seeds)')
        A('-' * 78)
        gg = runs.groupby('method')[['AUC', 'AP']].agg(['mean', 'std', 'size'])
        for meth, r in gg.sort_values(('AUC', 'mean'), ascending=False).iterrows():
            n = int(r[('AUC', 'size')])
            A(f'  {meth:<26}AUROC {r[("AUC","mean")]*100:6.2f}   '
              f'AP {r[("AP","mean")]*100:6.2f}   n={n}')
    A('')
    A('CAVEATS THAT TRAVEL WITH THESE NUMBERS')
    A('-' * 78)
    A('  * sd values are POPULATION sd (ddof=0); on n=3 this understates the sample sd')
    A('    by a factor of 1.22.')
    A('  * DeLong p-values condition on THESE trained models and carry no seed variance.')
    A('    The seed-level column in m3_tests_*.csv is the one that speaks to retraining.')
    A('  * the post-hoc head WIDTH was selected on test AUC; the reported recovery % is')
    A('    therefore a test-set maximum.')
    A('  * the test set is 50% prevalence by construction, so AP is optimistic relative')
    A('    to clinical prevalence.')
    A(f'  * held-out numbers exist only where a selection card from another dataset was')
    A(f'    applied (heldout_*.csv). Everything else was selected on the data it reports.')
    with open(path, 'w') as f:
        f.write('\n'.join(L) + '\n')
    return path


BUNDLE_DIR = f'/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
_stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M')
BUNDLE = f'{BUNDLE_DIR}/dl_results_{DATASET}_{RUN_VERSION}_{_stamp}.zip'

write_summary(f'{OUTPUT_DIR}/SUMMARY_{DATASET}.txt')

_skip_ext = () if INCLUDE_WEIGHTS else ('.pt', '.pth', '.ckpt')
_counts, _bytes = {}, 0
with zipfile.ZipFile(BUNDLE, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for root, _dirs, files in os.walk(OUTPUT_DIR):
        for fn in sorted(files):
            if fn.endswith(_skip_ext):
                continue
            full = os.path.join(root, fn)
            z.write(full, os.path.relpath(full, os.path.dirname(OUTPUT_DIR)))
            ext = os.path.splitext(fn)[1] or '(none)'
            _counts[ext] = _counts.get(ext, 0) + 1
            _bytes += os.path.getsize(full)

print(f'\nBUNDLE WRITTEN\n  {BUNDLE}')
print(f'  zipped size   : {_human(os.path.getsize(BUNDLE))}   '
      f'(raw {_human(_bytes)})')
_wmsg = 'INCLUDED' if INCLUDE_WEIGHTS else 'EXCLUDED (set INCLUDE_WEIGHTS=True to keep)'
print(f'  weights       : {_wmsg}')
print('\n  contents by type:')
for ext, n in sorted(_counts.items(), key=lambda kv: -kv[1]):
    print(f'    {ext:<10}{n:>5}')
_size = os.path.getsize(BUNDLE)
_LIMIT = 900 * 1024 ** 2       # Kaggle's browser download is unreliable well below 1 GB

# Kaggle serves /kaggle/working through the Output pane, but clicking Download on a large
# single file often does nothing at all -- no error, no progress. Splitting into volumes
# under the limit is what actually makes it downloadable, so do it automatically.
PARTS = [BUNDLE]
if _size > _LIMIT:
    print(f'\n  {_human(_size)} is too large for a reliable browser download — splitting.')
    PARTS = []
    with open(BUNDLE, 'rb') as src:
        i = 0
        while True:
            chunk = src.read(_LIMIT)
            if not chunk:
                break
            part = f'{BUNDLE}.part{i:02d}'
            with open(part, 'wb') as out:
                out.write(chunk)
            PARTS.append(part)
            i += 1
    os.remove(BUNDLE)          # keep only the parts, or the Output pane shows both
    print(f'  {len(PARTS)} parts written. Rejoin them locally with:')
    print(f'    cat {os.path.basename(BUNDLE)}.part* > {os.path.basename(BUNDLE)}')

print('\n  DOWNLOAD — try these in order:')
print('   1. Click each file below (works even when the Output pane button does not).')
try:
    from IPython.display import FileLink, display
    for p_ in PARTS:
        display(FileLink(os.path.relpath(p_, '/kaggle/working')
                         if p_.startswith('/kaggle/working') else p_))
except Exception as _e:
    print(f'   (FileLink unavailable: {type(_e).__name__})')
print('   2. If nothing downloads: Save Version -> "Save & Run All (Commit)". When it')
print('      finishes, open the finished version and use its Output tab. Committed')
print('      output downloads reliably; an interactive session\'s often does not.')
print('   3. Still stuck? Set INCLUDE_WEIGHTS=False and re-run this cell — that drops')
print('      the bundle to a few MB and every table still recomputes from the scores.')
if not INCLUDE_WEIGHTS:
    print('\n  NOTE weights excluded; every table still recomputes from the stored scores.')